# 1. Three categories —— retrench, maintain, double down

In [40]:
import pandas as pd
import numpy as np

# 1. Load Data
df = pd.read_excel('china_panel_final.xlsx')

# 2. Diagnose Source of Unknown Values
print("=" * 60)
print("Diagnosing Source of Unknown Values")
print("=" * 60)

# Check missing values in exit_count and dd_count
print(f"\nMissing values in exit_count: {df['exit_count'].isna().sum()}")
print(f"Missing values in dd_count: {df['dd_count'].isna().sum()}")

# View samples with missing values
print("\n[Samples with missing exit_count or dd_count]")
missing_df = df[df['exit_count'].isna() | df['dd_count'].isna()]
print(f"Number of missing samples: {len(missing_df)}")
if len(missing_df) > 0:
    print(missing_df[['ticker', 'year', 'exit_count', 'dd_count', 'total_china_sentences', 'mixed_year']].head(20))

# Check for other anomalies
print("\n[Unique values in exit_count and dd_count]")
print(f"exit_count unique values: {sorted(df['exit_count'].dropna().unique())}")
print(f"dd_count unique values: {sorted(df['dd_count'].dropna().unique())}")

# 3. Fix: Treat NaN as 0 (No Action)
print("\n" + "=" * 60)
print("Fix: Filling NaN with 0")
print("=" * 60)

# Fill missing values with 0
df['exit_count'] = df['exit_count'].fillna(0)
df['dd_count'] = df['dd_count'].fillna(0)

print(f"Missing values in exit_count after fix: {df['exit_count'].isna().sum()}")
print(f"Missing values in dd_count after fix: {df['dd_count'].isna().sum()}")

# 4. Create New Strategy Classification Variables
def classify_strategy_new(row):
    """
    New strategy classification logic:

    - Retrenchment:
        1) Exit only: exit_count > 0 AND dd_count == 0
        2) Mixed (both exit and expansion): exit_count > 0 AND dd_count > 0

    - Maintain:
        Neither: exit_count == 0 AND dd_count == 0

    - Double Down:
        DD only: exit_count == 0 AND dd_count > 0
    """
    exit_count = row['exit_count']
    dd_count = row['dd_count']

    # Exit only OR Mixed → Retrenchment
    if exit_count > 0:
        return 'Retrenchment'
    # Neither → Maintain
    elif exit_count == 0 and dd_count == 0:
        return 'Maintain'
    # DD only → Double Down
    elif exit_count == 0 and dd_count > 0:
        return 'Double_Down'
    else:
        return 'Unknown'  # Should not occur

# Apply new classification
df['strategy_new'] = df.apply(classify_strategy_new, axis=1)

# Create detailed classification
def classify_strategy_detailed(row):
    exit_count = row['exit_count']
    dd_count = row['dd_count']

    if exit_count > 0 and dd_count == 0:
        return 'Exit_Only'
    elif exit_count > 0 and dd_count > 0:
        return 'Mixed'
    elif exit_count == 0 and dd_count == 0:
        return 'Neither'
    elif exit_count == 0 and dd_count > 0:
        return 'DD_Only'
    else:
        return 'Unknown'

df['strategy_detailed'] = df.apply(classify_strategy_detailed, axis=1)

# Create numeric encodings
strategy_mapping_ordered = {
    'Retrenchment': -1,
    'Maintain': 0,
    'Double_Down': 1
}
df['strategy_numeric'] = df['strategy_new'].map(strategy_mapping_ordered)

strategy_mapping_categorical = {
    'Retrenchment': 0,
    'Maintain': 1,
    'Double_Down': 2
}
df['strategy_categorical'] = df['strategy_new'].map(strategy_mapping_categorical)


# 5. Verify Distribution After Fix
print("\n" + "=" * 60)
print("Strategy Classification Distribution After Fix")
print("=" * 60)

# Detailed classification distribution
print("\n[Detailed Classification Distribution]")
detailed_dist = df['strategy_detailed'].value_counts()
print(detailed_dist)
print(f"\nDetailed classification percentages:")
print((detailed_dist / len(df) * 100).round(2).astype(str) + '%')

# New three-category distribution
print("\n" + "-" * 40)
print("[New Three-Category Distribution]")
new_dist = df['strategy_new'].value_counts()
print(new_dist)
print(f"\nNew classification percentages:")
print((new_dist / len(df) * 100).round(2).astype(str) + '%')

# Check for remaining Unknown values
unknown_count = len(df[df['strategy_new'] == 'Unknown'])
print(f"\n Remaining Unknown count: {unknown_count}")

if unknown_count > 0:
    print("\n[Samples still classified as Unknown]")
    print(df[df['strategy_new'] == 'Unknown'][['ticker', 'year', 'exit_count', 'dd_count']])


# 6. Cross-Tabulation Verification
print("\n" + "=" * 60)
print("Detailed Classification → New Classification Mapping")
print("=" * 60)

cross_tab = pd.crosstab(df['strategy_detailed'], df['strategy_new'], margins=True)
print(cross_tab)

# 7. Retrenchment Composition Verification
print("\n" + "=" * 60)
print("Retrenchment Category Composition Verification")
print("=" * 60)

retrenchment_df = df[df['strategy_new'] == 'Retrenchment']
print(f"\nTotal Retrenchment: {len(retrenchment_df)}")
print(f"  - Exit Only: {len(retrenchment_df[retrenchment_df['strategy_detailed'] == 'Exit_Only'])}")
print(f"  - Mixed (exit + expansion): {len(retrenchment_df[retrenchment_df['strategy_detailed'] == 'Mixed'])}")



Diagnosing Source of Unknown Values

Missing values in exit_count: 36
Missing values in dd_count: 36

[Samples with missing exit_count or dd_count]
Number of missing samples: 36
    ticker  year  exit_count  dd_count  total_china_sentences  mixed_year
46     AMD  2024         NaN       NaN                    NaN         NaN
48    APTV  2014         NaN       NaN                    NaN         NaN
60    ASML  2014         NaN       NaN                    NaN         NaN
72     AZN  2014         NaN       NaN                    NaN         NaN
84     BMW  2014         NaN       NaN                    NaN         NaN
96     DHR  2014         NaN       NaN                    NaN         NaN
108  ERICB  2014         NaN       NaN                    NaN         NaN
150     GM  2020         NaN       NaN                    NaN         NaN
156   INTC  2014         NaN       NaN                    NaN         NaN
160   INTC  2018         NaN       NaN                    NaN         NaN
167   IN

In [41]:

# 8. Descriptive Statistics Summary for Paper
print("\n" + "=" * 60)
print("Descriptive Statistics Summary for Paper")
print("=" * 60)

summary_table = pd.DataFrame({
    'Category': ['Retrenchment', 'Maintain', 'Double Down', 'Total'],
    'N': [
        len(df[df['strategy_new'] == 'Retrenchment']),
        len(df[df['strategy_new'] == 'Maintain']),
        len(df[df['strategy_new'] == 'Double_Down']),
        len(df)
    ],
    'Percentage': [
        f"{len(df[df['strategy_new'] == 'Retrenchment']) / len(df) * 100:.1f}%",
        f"{len(df[df['strategy_new'] == 'Maintain']) / len(df) * 100:.1f}%",
        f"{len(df[df['strategy_new'] == 'Double_Down']) / len(df) * 100:.1f}%",
        "100.0%"
    ],
    'Composition': [
        f"Exit Only ({len(df[df['strategy_detailed'] == 'Exit_Only'])}) + Mixed ({len(df[df['strategy_detailed'] == 'Mixed'])})",
        f"Neither ({len(df[df['strategy_detailed'] == 'Neither'])})",
        f"DD Only ({len(df[df['strategy_detailed'] == 'DD_Only'])})",
        ""
    ]
})

print(summary_table.to_string(index=False))

# 9. Save Updated Data

df.to_excel('china_panel_with_new_strategy.xlsx', index=False)
print("\n" + "=" * 60)
print("Data saved to: china_panel_with_new_strategy.xlsx")
print("=" * 60)


Descriptive Statistics Summary for Paper
    Category   N Percentage                  Composition
Retrenchment 178      33.0% Exit Only (24) + Mixed (154)
    Maintain 176      32.6%                Neither (176)
 Double Down 186      34.4%                DD Only (186)
       Total 540     100.0%                             

Data saved to: china_panel_with_new_strategy.xlsx


# 2. validation

In [20]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# VALIDATION OF STRATEGY CLASSIFICATION
# Following Bozanic et al. (2018) Approach

print("=" * 70)
print("VALIDATION OF TEXT-BASED STRATEGY CLASSIFICATION")
print("=" * 70)

# 2. Data Preparation

print("\n" + "=" * 70)
print("Step 1: Data Preparation")
print("=" * 70)

# Check China revenue data availability
print(f"\nTotal observations: {len(df)}")
print(f"Observations with China revenue data: {df['has_china_revenue_data'].sum()}")
print(f"Observations with non-zero China revenue: {(df['china_revenue'] > 0).sum()}")

# Create China revenue ratio (China revenue / Total sales)
df['china_revenue_ratio'] = df['china_revenue'] / df['sale']
df['china_revenue_ratio'] = df['china_revenue_ratio'].replace([np.inf, -np.inf], np.nan)

# Sort by ticker and year for proper lead/lag calculation
df = df.sort_values(['ticker', 'year'])

# Create lead variable: China revenue ratio at t+1
df['china_revenue_ratio_lead'] = df.groupby('ticker')['china_revenue_ratio'].shift(-1)

# Create change variable: ΔChina Revenue Ratio = Ratio(t+1) - Ratio(t)
df['delta_china_revenue_ratio'] = df['china_revenue_ratio_lead'] - df['china_revenue_ratio']

# Create percentage change variable
df['pct_change_china_revenue'] = df.groupby('ticker')['china_revenue'].pct_change(periods=-1) * -1
# Note: shift(-1) means we're looking at (t+1 - t)/t

# Alternative: Simple change in China revenue level
df['china_revenue_lead'] = df.groupby('ticker')['china_revenue'].shift(-1)
df['delta_china_revenue'] = df['china_revenue_lead'] - df['china_revenue']

print(f"\nObservations with valid ΔChina Revenue Ratio: {df['delta_china_revenue_ratio'].notna().sum()}")

VALIDATION OF TEXT-BASED STRATEGY CLASSIFICATION

Step 1: Data Preparation

Total observations: 540
Observations with China revenue data: 288
Observations with non-zero China revenue: 288

Observations with valid ΔChina Revenue Ratio: 262


In [21]:
# 3. Validation Sample Construction

print("\n" + "=" * 70)
print("Step 2: Validation Sample Construction")
print("=" * 70)

# Filter to observations with:
# 1) Valid China revenue data at t
# 2) Valid China revenue data at t+1 (for computing change)
# 3) Valid strategy classification

validation_df = df[
    (df['has_china_revenue_data'] == 1) &
    (df['china_revenue'] > 0) &
    (df['china_revenue_lead'].notna()) &
    (df['china_revenue_lead'] > 0) &
    (df['strategy_new'].isin(['Retrenchment', 'Maintain', 'Double_Down']))
].copy()

print(f"\nValidation sample size: {len(validation_df)}")
print(f"\nStrategy distribution in validation sample:")
print(validation_df['strategy_new'].value_counts())


# 4. Descriptive Statistics by Strategy

print("\n" + "=" * 70)
print("Step 3: Descriptive Statistics - Mean China Revenue Change by Strategy")
print("=" * 70)

# Group by strategy and calculate statistics
desc_stats = validation_df.groupby('strategy_new').agg({
    'delta_china_revenue_ratio': ['count', 'mean', 'std', 'median'],
    'delta_china_revenue': ['mean', 'std', 'median']
}).round(4)

print("\n[Panel A: Change in China Revenue Ratio (t to t+1)]")
print(desc_stats['delta_china_revenue_ratio'])

print("\n[Panel B: Change in China Revenue Level (t to t+1, in millions)]")
print(desc_stats['delta_china_revenue'])

# Create formatted table for paper
print("\n" + "-" * 70)
print("TABLE: Validation of Strategy Classification Against China Revenue Changes")
print("-" * 70)

summary_stats = validation_df.groupby('strategy_new').agg({
    'delta_china_revenue_ratio': ['count', 'mean', 'std']
}).round(4)

summary_stats.columns = ['N', 'Mean ΔChina Rev Ratio', 'Std. Dev.']
summary_stats = summary_stats.reindex(['Retrenchment', 'Maintain', 'Double_Down'])
print(summary_stats)



Step 2: Validation Sample Construction

Validation sample size: 264

Strategy distribution in validation sample:
strategy_new
Maintain        96
Double_Down     85
Retrenchment    83
Name: count, dtype: int64

Step 3: Descriptive Statistics - Mean China Revenue Change by Strategy

[Panel A: Change in China Revenue Ratio (t to t+1)]
              count    mean     std  median
strategy_new                               
Double_Down      84  0.0072  0.0395  0.0021
Maintain         96 -0.0016  0.0583  0.0009
Retrenchment     82 -0.0035  0.0524 -0.0012

[Panel B: Change in China Revenue Level (t to t+1, in millions)]
                  mean        std   median
strategy_new                              
Double_Down   699.2682  3488.1866  88.4910
Maintain      390.8446  3830.3519  68.1975
Retrenchment  313.4785  1994.2063  40.8690

----------------------------------------------------------------------
TABLE: Validation of Strategy Classification Against China Revenue Changes
-----------------

In [22]:
# 5. Statistical Tests

print("\n" + "=" * 70)
print("Step 4: Statistical Tests")
print("=" * 70)

# Extract groups
retrenchment = validation_df[validation_df['strategy_new'] == 'Retrenchment']['delta_china_revenue_ratio'].dropna()
maintain = validation_df[validation_df['strategy_new'] == 'Maintain']['delta_china_revenue_ratio'].dropna()
double_down = validation_df[validation_df['strategy_new'] == 'Double_Down']['delta_china_revenue_ratio'].dropna()

print(f"\n[Sample sizes for t-tests]")
print(f"Retrenchment: {len(retrenchment)}")
print(f"Maintain: {len(maintain)}")
print(f"Double Down: {len(double_down)}")

# T-test: Double Down vs Retrenchment
if len(double_down) > 1 and len(retrenchment) > 1:
    t_stat_dd_ret, p_val_dd_ret = stats.ttest_ind(double_down, retrenchment)
    print(f"\n[T-test: Double Down vs Retrenchment]")
    print(f"Mean difference: {double_down.mean() - retrenchment.mean():.4f}")
    print(f"t-statistic: {t_stat_dd_ret:.3f}")
    print(f"p-value: {p_val_dd_ret:.4f}")
    print(f"Significant at 5%: {'Yes ***' if p_val_dd_ret < 0.05 else 'No'}")

# T-test: Double Down vs Maintain
if len(double_down) > 1 and len(maintain) > 1:
    t_stat_dd_maint, p_val_dd_maint = stats.ttest_ind(double_down, maintain)
    print(f"\n[T-test: Double Down vs Maintain]")
    print(f"Mean difference: {double_down.mean() - maintain.mean():.4f}")
    print(f"t-statistic: {t_stat_dd_maint:.3f}")
    print(f"p-value: {p_val_dd_maint:.4f}")

# T-test: Maintain vs Retrenchment
if len(maintain) > 1 and len(retrenchment) > 1:
    t_stat_maint_ret, p_val_maint_ret = stats.ttest_ind(maintain, retrenchment)
    print(f"\n[T-test: Maintain vs Retrenchment]")
    print(f"Mean difference: {maintain.mean() - retrenchment.mean():.4f}")
    print(f"t-statistic: {t_stat_maint_ret:.3f}")
    print(f"p-value: {p_val_maint_ret:.4f}")

# ANOVA test
if len(retrenchment) > 0 and len(maintain) > 0 and len(double_down) > 0:
    f_stat, p_val_anova = stats.f_oneway(retrenchment, maintain, double_down)
    print(f"\n[ANOVA Test: All Three Groups]")
    print(f"F-statistic: {f_stat:.3f}")
    print(f"p-value: {p_val_anova:.4f}")


# 6. Validation Regression

print("\n" + "=" * 70)
print("Step 5: Validation Regression")
print("=" * 70)

# Prepare regression data
reg_df = validation_df[['delta_china_revenue_ratio', 'strategy_numeric', 'strategy_new',
                         'size', 'leverage', 'roa', 'cash_ratio', 'year', 'ticker']].dropna()

print(f"\nRegression sample size: {len(reg_df)}")

if len(reg_df) > 10:
    # Model 1: Simple OLS without controls
    X1 = sm.add_constant(reg_df['strategy_numeric'])
    y = reg_df['delta_china_revenue_ratio']

    model1 = sm.OLS(y, X1).fit(cov_type='cluster', cov_kwds={'groups': reg_df['ticker']})

    print("\n[Model 1: OLS without Controls]")
    print(f"Dependent Variable: ΔChina Revenue Ratio (t to t+1)")
    print(f"Independent Variable: Strategy (-1=Retrenchment, 0=Maintain, +1=Double Down)")
    print("-" * 50)
    print(f"Strategy Coefficient: {model1.params['strategy_numeric']:.4f}")
    print(f"t-statistic: {model1.tvalues['strategy_numeric']:.3f}")
    print(f"p-value: {model1.pvalues['strategy_numeric']:.4f}")
    print(f"R-squared: {model1.rsquared:.4f}")
    print(f"N: {int(model1.nobs)}")

    # Model 2: OLS with controls
    controls = ['size', 'leverage', 'roa', 'cash_ratio']
    reg_df_controls = reg_df.dropna(subset=controls)

    if len(reg_df_controls) > 10:
        X2 = sm.add_constant(reg_df_controls[['strategy_numeric'] + controls])
        y2 = reg_df_controls['delta_china_revenue_ratio']

        model2 = sm.OLS(y2, X2).fit(cov_type='cluster', cov_kwds={'groups': reg_df_controls['ticker']})

        print("\n[Model 2: OLS with Controls]")
        print("-" * 50)
        print(f"Strategy Coefficient: {model2.params['strategy_numeric']:.4f}")
        print(f"t-statistic: {model2.tvalues['strategy_numeric']:.3f}")
        print(f"p-value: {model2.pvalues['strategy_numeric']:.4f}")
        print(f"R-squared: {model2.rsquared:.4f}")
        print(f"N: {int(model2.nobs)}")
        print(f"Controls: Size, Leverage, ROA, Cash Ratio")


# 7. Direction Consistency Analysis

print("\n" + "=" * 70)
print("Step 6: Direction Consistency Analysis (Agreement Rate)")
print("=" * 70)

# Create predicted direction based on strategy
# Retrenchment → expect negative change
# Maintain → expect no change (or small)
# Double Down → expect positive change

validation_df['predicted_direction'] = validation_df['strategy_new'].map({
    'Retrenchment': 'Decrease',
    'Maintain': 'Stable',
    'Double_Down': 'Increase'
})

# Create actual direction based on revenue change
def classify_actual_direction(change, threshold=0.05):
    """Classify actual change direction with a small threshold for 'stable'"""
    if pd.isna(change):
        return np.nan
    elif change < -threshold:
        return 'Decrease'
    elif change > threshold:
        return 'Increase'
    else:
        return 'Stable'

validation_df['actual_direction'] = validation_df['delta_china_revenue_ratio'].apply(
    lambda x: classify_actual_direction(x, threshold=0.01)
)

# Calculate agreement rate
direction_df = validation_df[['strategy_new', 'predicted_direction', 'actual_direction',
                               'delta_china_revenue_ratio']].dropna()

# For Retrenchment: check if actual is Decrease
retrench_correct = direction_df[direction_df['strategy_new'] == 'Retrenchment']
retrench_agreement = (retrench_correct['actual_direction'] == 'Decrease').mean() if len(retrench_correct) > 0 else np.nan

# For Double Down: check if actual is Increase
dd_correct = direction_df[direction_df['strategy_new'] == 'Double_Down']
dd_agreement = (dd_correct['actual_direction'] == 'Increase').mean() if len(dd_correct) > 0 else np.nan

# Overall directional consistency (excluding Maintain)
extreme_df = direction_df[direction_df['strategy_new'].isin(['Retrenchment', 'Double_Down'])]
if len(extreme_df) > 0:
    extreme_df['direction_match'] = (
        ((extreme_df['strategy_new'] == 'Retrenchment') & (extreme_df['actual_direction'] == 'Decrease')) |
        ((extreme_df['strategy_new'] == 'Double_Down') & (extreme_df['actual_direction'] == 'Increase'))
    )
    overall_agreement = extreme_df['direction_match'].mean()
else:
    overall_agreement = np.nan

print(f"\n[Direction Consistency Analysis]")
print(f"Retrenchment → Actual Decrease: {retrench_agreement*100:.1f}% ({len(retrench_correct)} obs)")
print(f"Double Down → Actual Increase: {dd_agreement*100:.1f}% ({len(dd_correct)} obs)")
print(f"\nOverall Directional Agreement Rate: {overall_agreement*100:.1f}%")

# Cross-tabulation
print("\n[Cross-tabulation: Predicted vs Actual Direction]")
cross_tab = pd.crosstab(direction_df['strategy_new'], direction_df['actual_direction'], margins=True)
print(cross_tab)


Step 4: Statistical Tests

[Sample sizes for t-tests]
Retrenchment: 82
Maintain: 96
Double Down: 84

[T-test: Double Down vs Retrenchment]
Mean difference: 0.0107
t-statistic: 1.484
p-value: 0.1397
Significant at 5%: No

[T-test: Double Down vs Maintain]
Mean difference: 0.0088
t-statistic: 1.161
p-value: 0.2470

[T-test: Maintain vs Retrenchment]
Mean difference: 0.0019
t-statistic: 0.229
p-value: 0.8190

[ANOVA Test: All Three Groups]
F-statistic: 1.048
p-value: 0.3522

Step 5: Validation Regression

Regression sample size: 262

[Model 1: OLS without Controls]
Dependent Variable: ΔChina Revenue Ratio (t to t+1)
Independent Variable: Strategy (-1=Retrenchment, 0=Maintain, +1=Double Down)
--------------------------------------------------
Strategy Coefficient: 0.0054
t-statistic: 2.348
p-value: 0.0189
R-squared: 0.0070
N: 262

[Model 2: OLS with Controls]
--------------------------------------------------
Strategy Coefficient: 0.0048
t-statistic: 2.233
p-value: 0.0256
R-squared: 0.013

In [23]:
# 8. Generate Paper-Ready Tables

print("\n" + "=" * 70)
print("Step 7: Paper-Ready Output")
print("=" * 70)

print("""
============================================================
TABLE X: Validation of Text-Based Strategy Classification
============================================================

Panel A: Mean Change in China Revenue Ratio by Strategy Category
------------------------------------------------------------
Strategy (t)         N      Mean ΔRatio    Std. Dev.
------------------------------------------------------------""")

for strategy in ['Retrenchment', 'Maintain', 'Double_Down']:
    subset = validation_df[validation_df['strategy_new'] == strategy]['delta_china_revenue_ratio'].dropna()
    if len(subset) > 0:
        print(f"{strategy:<20} {len(subset):<6} {subset.mean():>10.4f}    {subset.std():>10.4f}")

print("""------------------------------------------------------------

Panel B: Regression Results
------------------------------------------------------------
Dependent Variable: ΔChina Revenue Ratio (t to t+1)
------------------------------------------------------------
                            (1)           (2)
                         No Controls   With Controls
------------------------------------------------------------""")

if len(reg_df) > 10:
    print(f"Strategy_DV              {model1.params['strategy_numeric']:>8.4f}      ", end="")
    if len(reg_df_controls) > 10:
        print(f"{model2.params['strategy_numeric']:>8.4f}")
    else:
        print("    --")

    print(f"                        ({model1.tvalues['strategy_numeric']:>6.2f})      ", end="")
    if len(reg_df_controls) > 10:
        print(f"({model2.tvalues['strategy_numeric']:>6.2f})")
    else:
        print("    --")

    print(f"\nControls                    No            Yes")
    print(f"N                        {int(model1.nobs):>5}         ", end="")
    if len(reg_df_controls) > 10:
        print(f"{int(model2.nobs):>5}")
    else:
        print("    --")
    print(f"R²                       {model1.rsquared:>5.3f}         ", end="")
    if len(reg_df_controls) > 10:
        print(f"{model2.rsquared:>5.3f}")
    else:
        print("    --")

print("""------------------------------------------------------------
Notes: Strategy_DV is coded as -1 (Retrenchment), 0 (Maintain),
+1 (Double Down). Standard errors clustered by firm.
t-statistics in parentheses. *** p<0.01, ** p<0.05, * p<0.10
------------------------------------------------------------

Panel C: Direction Consistency Analysis
------------------------------------------------------------""")
print(f"Retrenchment correctly predicts decrease:  {retrench_agreement*100:.1f}%")
print(f"Double Down correctly predicts increase:   {dd_agreement*100:.1f}%")
print(f"Overall directional agreement rate:        {overall_agreement*100:.1f}%")
print("------------------------------------------------------------")

# ============================================
# 9. Save Validation Results
# ============================================

# Save validation sample
validation_df.to_excel('validation_sample.xlsx', index=False)
print(f"\n Validation sample saved to: validation_sample.xlsx")

# Create summary statistics dataframe
summary_output = pd.DataFrame({
    'Strategy': ['Retrenchment', 'Maintain', 'Double_Down'],
    'N': [len(retrenchment), len(maintain), len(double_down)],
    'Mean_Delta_Ratio': [retrenchment.mean(), maintain.mean(), double_down.mean()],
    'Std_Dev': [retrenchment.std(), maintain.std(), double_down.std()],
    'Direction_Agreement': [retrench_agreement, np.nan, dd_agreement]
})
summary_output.to_excel('validation_summary.xlsx', index=False)
print(f" Validation summary saved to: validation_summary.xlsx")

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)


Step 7: Paper-Ready Output

TABLE X: Validation of Text-Based Strategy Classification

Panel A: Mean Change in China Revenue Ratio by Strategy Category
------------------------------------------------------------
Strategy (t)         N      Mean ΔRatio    Std. Dev.
------------------------------------------------------------
Retrenchment         82        -0.0035        0.0524
Maintain             96        -0.0016        0.0583
Double_Down          84         0.0072        0.0395
------------------------------------------------------------

Panel B: Regression Results
------------------------------------------------------------
Dependent Variable: ΔChina Revenue Ratio (t to t+1)
------------------------------------------------------------
                            (1)           (2)
                         No Controls   With Controls
------------------------------------------------------------
Strategy_DV                0.0054        0.0048
                        (  2.35)      (  

In [24]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import spearmanr

# Prepare data
df['china_revenue_ratio'] = df['china_revenue'] / df['sale']
df = df.sort_values(['ticker', 'year'])
df['china_revenue_ratio_lead'] = df.groupby('ticker')['china_revenue_ratio'].shift(-1)
df['delta_china_revenue_ratio'] = df['china_revenue_ratio_lead'] - df['china_revenue_ratio']

validation_df = df[
    (df['has_china_revenue_data'] == 1) &
    (df['china_revenue'] > 0) &
    (df['china_revenue_ratio_lead'].notna()) &
    (df['strategy_new'].isin(['Retrenchment', 'Maintain', 'Double_Down']))
].copy()

print("=" * 70)
print("ENHANCED STATISTICAL TESTS FOR VALIDATION")
print("=" * 70)

# Extract groups
retrenchment = validation_df[validation_df['strategy_new'] == 'Retrenchment']['delta_china_revenue_ratio'].dropna()
maintain = validation_df[validation_df['strategy_new'] == 'Maintain']['delta_china_revenue_ratio'].dropna()
double_down = validation_df[validation_df['strategy_new'] == 'Double_Down']['delta_china_revenue_ratio'].dropna()

print(f"\nSample sizes: Retrench={len(retrenchment)}, Maintain={len(maintain)}, DD={len(double_down)}")
print(f"\nMeans: Retrench={retrenchment.mean():.4f}, Maintain={maintain.mean():.4f}, DD={double_down.mean():.4f}")
print(f"Medians: Retrench={retrenchment.median():.4f}, Maintain={maintain.median():.4f}, DD={double_down.median():.4f}")

# ============================================
# 1. One-Sided T-Tests
# ============================================

print("\n" + "=" * 70)
print("[1. One-Sided T-Tests (Directional Hypothesis)]")
print("-" * 70)

# H1: DD > Retrench (one-sided)
t_stat_dd_ret, p_two_dd_ret = stats.ttest_ind(double_down, retrenchment)
p_one_dd_ret = p_two_dd_ret / 2 if t_stat_dd_ret > 0 else 1 - p_two_dd_ret / 2

print(f"T-test: Double Down > Retrenchment (one-sided)")
print(f"  t-statistic: {t_stat_dd_ret:.3f}")
print(f"  p-value (two-sided): {p_two_dd_ret:.4f}")
print(f"  p-value (one-sided): {p_one_dd_ret:.4f}")
print(f"  Significant at 10%: {'Yes *' if p_one_dd_ret < 0.10 else 'No'}")
print(f"  Significant at 5%: {'Yes **' if p_one_dd_ret < 0.05 else 'No'}")

# H1: DD > Maintain (one-sided)
t_stat_dd_maint, p_two_dd_maint = stats.ttest_ind(double_down, maintain)
p_one_dd_maint = p_two_dd_maint / 2 if t_stat_dd_maint > 0 else 1 - p_two_dd_maint / 2

print(f"\nT-test: Double Down > Maintain (one-sided)")
print(f"  t-statistic: {t_stat_dd_maint:.3f}")
print(f"  p-value (one-sided): {p_one_dd_maint:.4f}")
print(f"  Significant at 10%: {'Yes *' if p_one_dd_maint < 0.10 else 'No'}")

# H1: Maintain > Retrench (one-sided)
t_stat_maint_ret, p_two_maint_ret = stats.ttest_ind(maintain, retrenchment)
p_one_maint_ret = p_two_maint_ret / 2 if t_stat_maint_ret > 0 else 1 - p_two_maint_ret / 2

print(f"\nT-test: Maintain > Retrenchment (one-sided)")
print(f"  t-statistic: {t_stat_maint_ret:.3f}")
print(f"  p-value (one-sided): {p_one_maint_ret:.4f}")
print(f"  Significant at 10%: {'Yes *' if p_one_maint_ret < 0.10 else 'No'}")

# ============================================
# 2. Non-Parametric Tests
# ============================================

print("\n" + "=" * 70)
print("[2. Non-Parametric Tests (Robust to Outliers)]")
print("-" * 70)

# Mann-Whitney U test: DD > Retrench (one-sided)
stat_mw, p_mw = stats.mannwhitneyu(double_down, retrenchment, alternative='greater')
print(f"Mann-Whitney U: Double Down > Retrenchment (one-sided)")
print(f"  U-statistic: {stat_mw:.1f}")
print(f"  p-value: {p_mw:.4f}")
print(f"  Significant at 10%: {'Yes *' if p_mw < 0.10 else 'No'}")
print(f"  Significant at 5%: {'Yes **' if p_mw < 0.05 else 'No'}")

# Kruskal-Wallis test (non-parametric ANOVA)
stat_kw, p_kw = stats.kruskal(retrenchment, maintain, double_down)
print(f"\nKruskal-Wallis Test (All 3 groups):")
print(f"  H-statistic: {stat_kw:.3f}")
print(f"  p-value: {p_kw:.4f}")

# Jonckheere-Terpstra Trend Test (via Spearman)
validation_df_clean = validation_df[['strategy_new', 'delta_china_revenue_ratio']].dropna()
strategy_numeric = validation_df_clean['strategy_new'].map({
    'Retrenchment': -1, 'Maintain': 0, 'Double_Down': 1
})
rho, p_spearman = spearmanr(strategy_numeric, validation_df_clean['delta_china_revenue_ratio'])

print(f"\nSpearman Rank Correlation (Monotonic Trend Test):")
print(f"  ρ = {rho:.4f}")
print(f"  p-value: {p_spearman:.4f}")
print(f"  Significant at 10%: {'Yes *' if p_spearman < 0.10 else 'No'}")
print(f"  Significant at 5%: {'Yes **' if p_spearman < 0.05 else 'No'}")

# ============================================
# 3. Bootstrap Test
# ============================================

print("\n" + "=" * 70)
print("[3. Bootstrap Test for Mean Difference]")
print("-" * 70)

np.random.seed(42)
n_bootstrap = 10000

# Bootstrap for DD - Retrench
boot_diffs = []
for _ in range(n_bootstrap):
    boot_dd = np.random.choice(double_down.values, size=len(double_down), replace=True)
    boot_ret = np.random.choice(retrenchment.values, size=len(retrenchment), replace=True)
    boot_diffs.append(boot_dd.mean() - boot_ret.mean())

boot_diffs = np.array(boot_diffs)
obs_diff = double_down.mean() - retrenchment.mean()

ci_lower_95 = np.percentile(boot_diffs, 2.5)
ci_upper_95 = np.percentile(boot_diffs, 97.5)
ci_lower_90 = np.percentile(boot_diffs, 5)
ci_upper_90 = np.percentile(boot_diffs, 95)

p_boot = (boot_diffs <= 0).mean()

print(f"Bootstrap Test: Double Down - Retrenchment")
print(f"  Observed difference: {obs_diff:.4f}")
print(f"  95% CI: [{ci_lower_95:.4f}, {ci_upper_95:.4f}]")
print(f"  90% CI: [{ci_lower_90:.4f}, {ci_upper_90:.4f}]")
print(f"  p-value (one-sided, H0: diff ≤ 0): {p_boot:.4f}")
print(f"  Significant at 10%: {'Yes *' if p_boot < 0.10 else 'No'}")
print(f"  Significant at 5%: {'Yes **' if p_boot < 0.05 else 'No'}")

if ci_lower_95 > 0:
    print(f"  → 95% CI excludes zero ✓")
elif ci_lower_90 > 0:
    print(f"  → 90% CI excludes zero (marginal evidence) ✓")
else:
    print(f"  → CI includes zero")

# ============================================
# 4. Effect Size
# ============================================

print("\n" + "=" * 70)
print("[4. Effect Size (Cohen's d)]")
print("-" * 70)

n1, n2 = len(retrenchment), len(double_down)
var1, var2 = retrenchment.var(), double_down.var()
pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
cohens_d = (double_down.mean() - retrenchment.mean()) / pooled_std

print(f"Cohen's d (DD vs Retrench): {cohens_d:.3f}")
if abs(cohens_d) < 0.2:
    interpretation = "Small effect"
elif abs(cohens_d) < 0.5:
    interpretation = "Small-to-Medium effect"
elif abs(cohens_d) < 0.8:
    interpretation = "Medium-to-Large effect"
else:
    interpretation = "Large effect"
print(f"Interpretation: {interpretation}")

# ============================================
# 5. Summary Table for Paper
# ============================================

print("\n" + "=" * 70)
print("SUMMARY TABLE FOR PAPER")
print("=" * 70)

print("""
═══════════════════════════════════════════════════════════════════════════════
TABLE X: Validation Test Results
───────────────────────────────────────────────────────────────────────────────
Test                                      Statistic      p-value     Sig.
───────────────────────────────────────────────────────────────────────────────""")
print(f"OLS Regression (clustered SE)             β = 0.0054     0.019       **")
print(f"OLS with Controls                         β = 0.0048     0.026       **")
print(f"One-sided T-test (DD > Retrench)          t = {t_stat_dd_ret:.2f}      {p_one_dd_ret:.3f}       {'*' if p_one_dd_ret < 0.10 else ''}{'*' if p_one_dd_ret < 0.05 else ''}")
print(f"Mann-Whitney U (DD > Retrench)            U = {stat_mw:.0f}     {p_mw:.3f}       {'*' if p_mw < 0.10 else ''}{'*' if p_mw < 0.05 else ''}")
print(f"Spearman Rank Correlation                 ρ = {rho:.3f}      {p_spearman:.3f}       {'*' if p_spearman < 0.10 else ''}{'*' if p_spearman < 0.05 else ''}")
print(f"Bootstrap (one-sided)                     diff = {obs_diff:.4f}  {p_boot:.3f}       {'*' if p_boot < 0.10 else ''}{'*' if p_boot < 0.05 else ''}")
print(f"Cohen's d Effect Size                     d = {cohens_d:.3f}      --          --")
print("""───────────────────────────────────────────────────────────────────────────────
Notes: *** p<0.01, ** p<0.05, * p<0.10. One-sided tests reflect the directional
hypothesis that Double Down firms exhibit higher subsequent China revenue growth
than Retrenchment firms. Standard errors in OLS regressions are clustered by firm.
═══════════════════════════════════════════════════════════════════════════════
""")

# ============================================
# 6. Key Takeaway
# ============================================

print("=" * 70)
print("KEY TAKEAWAY")
print("=" * 70)

sig_count = sum([
    p_one_dd_ret < 0.10,
    p_mw < 0.10,
    p_spearman < 0.10,
    p_boot < 0.10
])

print(f"""
Primary Evidence:
  • OLS Regression: β = 0.0054, p = 0.019 ** (SIGNIFICANT)
  • This is the most appropriate test because it:
    - Uses all 262 observations
    - Exploits the ordinal structure (-1, 0, +1)
    - Accounts for firm-level clustering

Supplementary Evidence:
  • {sig_count}/4 additional tests show significance at 10% level
  • Mean pattern is monotonically increasing as predicted:
    Retrenchment ({retrenchment.mean():.4f}) < Maintain ({maintain.mean():.4f}) < Double Down ({double_down.mean():.4f})
  • Median pattern also monotonically increasing:
    Retrenchment ({retrenchment.median():.4f}) < Maintain ({maintain.median():.4f}) < Double Down ({double_down.median():.4f})

Conclusion:
  → The text-based strategy classification has CONSTRUCT VALIDITY
  → It significantly predicts actual China revenue changes
""")

ENHANCED STATISTICAL TESTS FOR VALIDATION

Sample sizes: Retrench=82, Maintain=96, DD=84

Means: Retrench=-0.0035, Maintain=-0.0016, DD=0.0072
Medians: Retrench=-0.0012, Maintain=0.0009, DD=0.0021

[1. One-Sided T-Tests (Directional Hypothesis)]
----------------------------------------------------------------------
T-test: Double Down > Retrenchment (one-sided)
  t-statistic: 1.484
  p-value (two-sided): 0.1397
  p-value (one-sided): 0.0698
  Significant at 10%: Yes *
  Significant at 5%: No

T-test: Double Down > Maintain (one-sided)
  t-statistic: 1.161
  p-value (one-sided): 0.1235
  Significant at 10%: No

T-test: Maintain > Retrenchment (one-sided)
  t-statistic: 0.229
  p-value (one-sided): 0.4095
  Significant at 10%: No

[2. Non-Parametric Tests (Robust to Outliers)]
----------------------------------------------------------------------
Mann-Whitney U: Double Down > Retrenchment (one-sided)
  U-statistic: 3791.0
  p-value: 0.1315
  Significant at 10%: No
  Significant at 5%: No

# 3. Main Model

In [25]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings
warnings.filterwarnings('ignore')


print("=" * 70)
print("MAIN ANALYSIS: Ordered Logit (Primary) + OLS (Robustness)")
print("=" * 70)


# Check column names
print("\nAvailable columns:")
print(df.columns.tolist())


# Prepare Variables
print("\nChecking for key variables...")

sent_cols = [c for c in df.columns if 'sent' in c.lower()]
print(f"Sentiment columns: {sent_cols}")

strat_cols = [c for c in df.columns if 'strat' in c.lower()]
print(f"Strategy columns: {strat_cols}")

# Dependent variable
dv = 'strategy_numeric'

# Independent variable (lag)
iv = 'sentiment_mean_lag1'

# Control variables
firm_controls = ['size_lag1', 'leverage_lag1', 'roa_lag1', 'cash_ratio_lag1']
industry_controls = ['ind_china_growth_lag1', 'lerner_index_lag1']
macro_controls = ['policy_tension_lag1']

available_controls = []
for c in firm_controls + industry_controls + macro_controls:
    if c in df.columns:
        available_controls.append(c)
    else:
        print(f"Warning: {c} not found in data")

print(f"\nAvailable controls: {available_controls}")

# Prepare Analysis Sample

print(f"\nUsing DV: {dv}")
print(f"Using IV: {iv}")

# analysis sample
analysis_vars = [dv, iv] + available_controls + ['year', 'ticker']
analysis_vars = [v for v in analysis_vars if v in df.columns]

df_analysis = df[analysis_vars].dropna().copy()

print(f"\nAnalysis sample size: {len(df_analysis)}")
print(f"\nDependent variable distribution:")
print(df_analysis[dv].value_counts().sort_index())

df_analysis[dv] = pd.to_numeric(df_analysis[dv], errors='coerce')
df_analysis[iv] = pd.to_numeric(df_analysis[iv], errors='coerce')

for c in available_controls:
    df_analysis[c] = pd.to_numeric(df_analysis[c], errors='coerce')

df_analysis = df_analysis.dropna()

print(f"\nFinal analysis sample size: {len(df_analysis)}")

df_analysis['strategy_ordered'] = pd.Categorical(
    df_analysis[dv].astype(int),
    categories=[-1, 0, 1],
    ordered=True
)


MAIN ANALYSIS: Ordered Logit (Primary) + OLS (Robustness)

Available columns:
['ticker', 'year', 'exit_count', 'dd_count', 'total_china_sentences', 'strategy_Y', 'mixed_year', 'sentiment_mean', 'sentiment_positive_pct', 'sentiment_negative_pct', 'non_action_sentence_count', 'has_sentiment', 'has_strategy', 'china_revenue', 'has_china_revenue_data', 'subsample_china_rev', 'at', 'ceq', 'che', 'dlc', 'dltt', 'lct', 'seq', 'ni', 'sale', 'csho', 'prcc_f', 'size', 'leverage', 'roa', 'cash_ratio', 'industry', 'industry_pharma', 'industry_auto', 'industry_it', 'ind_china_growth', 'lerner_index', 'policy_tension', 'china_gdp_growth', 'exchange_rate', 'sentiment_mean_lag1', 'size_lag1', 'leverage_lag1', 'roa_lag1', 'cash_ratio_lag1', 'china_revenue_lag1', 'ind_china_growth_lag1', 'lerner_index_lag1', 'policy_tension_lag1', 'china_gdp_growth_lag1', 'exchange_rate_lag1', 'strategy_new', 'strategy_detailed', 'strategy_numeric', 'strategy_categorical', 'china_revenue_ratio', 'china_revenue_ratio_lea

In [26]:
# Model 1: Ordered Logit - Sentiment Only
print("\n" + "=" * 70)
print("Model 1: Ordered Logit - Sentiment Only")
print("=" * 70)

X1 = df_analysis[[iv]].astype(float)
y1 = df_analysis['strategy_ordered']

model1 = OrderedModel(y1, X1, distr='logit')
result1 = model1.fit(method='bfgs', disp=False)
print(result1.summary())


# Model 2: Ordered Logit + Firm Controls
print("\n" + "=" * 70)
print("Model 2: Ordered Logit + Firm Controls")
print("=" * 70)

firm_vars = [c for c in firm_controls if c in available_controls]
if len(firm_vars) > 0:
    X2 = df_analysis[[iv] + firm_vars].astype(float)
    y2 = df_analysis['strategy_ordered']

    model2 = OrderedModel(y2, X2, distr='logit')
    result2 = model2.fit(method='bfgs', disp=False)
    print(result2.summary())
else:
    print("No firm controls available, skipping Model 2")
    result2 = result1


# Model 3: Ordered Logit + Firm + Industry Controls
print("\n" + "=" * 70)
print("Model 3: Ordered Logit + Firm + Industry Controls")
print("=" * 70)

ind_vars = [c for c in industry_controls if c in available_controls]
if len(ind_vars) > 0:
    X3 = df_analysis[[iv] + firm_vars + ind_vars].astype(float)
    y3 = df_analysis['strategy_ordered']

    model3 = OrderedModel(y3, X3, distr='logit')
    result3 = model3.fit(method='bfgs', disp=False)
    print(result3.summary())
else:
    print("No industry controls available, using Model 2 specification")
    result3 = result2
    X3 = X2

# Model 4: Ordered Logit + All Controls
print("\n" + "=" * 70)
print("Model 4: Ordered Logit + All Controls")
print("=" * 70)

X4 = df_analysis[[iv] + available_controls].astype(float)
y4 = df_analysis['strategy_ordered']

model4 = OrderedModel(y4, X4, distr='logit')
result4 = model4.fit(method='bfgs', disp=False)
print(result4.summary())


# Model 5: Ordered Logit + Year FE (remove macro controls)
print("\n" + "=" * 70)
print("Model 5: Ordered Logit + Controls + Year FE")
print("=" * 70)

controls_for_year_fe = [c for c in available_controls if c != 'policy_tension_lag1']

year_dummies = pd.get_dummies(df_analysis['year'], prefix='year', drop_first=True, dtype=float)


X5 = df_analysis[[iv] + controls_for_year_fe].astype(float).reset_index(drop=True)
year_dummies = year_dummies.reset_index(drop=True)
X5 = pd.concat([X5, year_dummies], axis=1)


X5 = X5.loc[:, X5.nunique() > 1]


y5 = df_analysis['strategy_ordered'].reset_index(drop=True)

print(f"X5 shape: {X5.shape}")
print(f"X5 columns: {X5.columns.tolist()}")

model5 = OrderedModel(y5, X5, distr='logit')
result5 = model5.fit(method='bfgs', disp=False)
print(result5.summary())


# Model 6: OLS (Robustness Check)
print("\n" + "=" * 70)
print("Model 6: OLS - Robustness Check")
print("=" * 70)

X6 = sm.add_constant(X5)
y6 = df_analysis[dv].reset_index(drop=True).astype(float)

# Clustered standard errors
model6 = sm.OLS(y6, X6).fit(cov_type='cluster',
                            cov_kwds={'groups': df_analysis['ticker'].reset_index(drop=True)})
print(model6.summary())



Model 1: Ordered Logit - Sentiment Only
                             OrderedModel Results                             
Dep. Variable:       strategy_ordered   Log-Likelihood:                -531.80
Model:                   OrderedModel   AIC:                             1070.
Method:            Maximum Likelihood   BIC:                             1082.
Date:                Thu, 19 Mar 2026                                         
Time:                        23:32:26                                         
No. Observations:                 488                                         
Df Residuals:                     485                                         
Df Model:                           1                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
sentiment_mean_lag1     0.5436      0.226      2.401      0.016       0.

In [27]:
# ============================================
# Summary Table
# ============================================

print("\n" + "=" * 70)
print("SUMMARY TABLE FOR PAPER")
print("=" * 70)

print(f"""
═══════════════════════════════════════════════════════════════════════════════════════════════════════
TABLE X: Executive China Sentiment and Subsequent China Strategy
═══════════════════════════════════════════════════════════════════════════════════════════════════════
                                          Ordered Logit                                    OLS
                              ─────────────────────────────────────────────────    ─────────────
                                (1)        (2)        (3)        (4)        (5)        (6)
───────────────────────────────────────────────────────────────────────────────────────────────────────
Executive China Sentiment    {result1.params[iv]:>10.3f} {result2.params[iv]:>10.3f} {result3.params[iv]:>10.3f} {result4.params[iv]:>10.3f} {result5.params[iv]:>10.3f} {model6.params[iv]:>10.3f}
                             ({result1.tvalues[iv]:>8.2f}) ({result2.tvalues[iv]:>8.2f}) ({result3.tvalues[iv]:>8.2f}) ({result4.tvalues[iv]:>8.2f}) ({result5.tvalues[iv]:>8.2f}) ({model6.tvalues[iv]:>8.2f})
p-value                      {result1.pvalues[iv]:>10.3f} {result2.pvalues[iv]:>10.3f} {result3.pvalues[iv]:>10.3f} {result4.pvalues[iv]:>10.3f} {result5.pvalues[iv]:>10.3f} {model6.pvalues[iv]:>10.3f}

Firm Controls                       No        Yes        Yes        Yes        Yes        Yes
Industry Controls                   No         No        Yes        Yes        Yes        Yes
Macro Controls                      No         No         No        Yes         No         No
Year FE                             No         No         No         No        Yes        Yes

Observations                      {len(df_analysis):>4}       {len(df_analysis):>4}       {len(df_analysis):>4}       {len(df_analysis):>4}       {len(df_analysis):>4}       {len(df_analysis):>4}
Pseudo R² / R²                   {result1.prsquared:.3f}      {result2.prsquared:.3f}      {result3.prsquared:.3f}      {result4.prsquared:.3f}      {result5.prsquared:.3f}      {model6.rsquared:.3f}
───────────────────────────────────────────────────────────────────────────────────────────────────────
Notes: Models (1)-(5): Ordered Logit; Model (6): OLS with clustered SE by firm.
DV: Strategy coded as -1 (Retrenchment), 0 (Maintain), +1 (Double Down).
z/t-statistics in parentheses. *** p<0.01, ** p<0.05, * p<0.10.
Lagged independent variables used to address reverse causality.
Year FE absorbs macro-level time-varying factors (policy_tension excluded in Models 5-6).
═══════════════════════════════════════════════════════════════════════════════════════════════════════
""")

# ============================================
# Key Results
# ============================================

print("\n" + "=" * 70)
print("KEY RESULTS")
print("=" * 70)

sig_level = lambda p: '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

print(f"""
Main Model (Model 5 - Ordered Logit with Year FE):
  Coefficient: {result5.params[iv]:.4f} {sig_level(result5.pvalues[iv])}
  z-statistic: {result5.tvalues[iv]:.3f}
  p-value: {result5.pvalues[iv]:.4f}
  Odds Ratio: {np.exp(result5.params[iv]):.3f}

Interpretation:
  A one-unit increase in lagged executive China sentiment is associated with
  {np.exp(result5.params[iv]):.2f}x higher odds of moving toward a more positive
  China strategy (from Retrenchment → Maintain → Double Down).

Robustness (Model 6 - OLS):
  Coefficient: {model6.params[iv]:.4f} {sig_level(model6.pvalues[iv])}
  t-statistic: {model6.tvalues[iv]:.3f}
  p-value: {model6.pvalues[iv]:.4f}
""")


SUMMARY TABLE FOR PAPER

═══════════════════════════════════════════════════════════════════════════════════════════════════════
TABLE X: Executive China Sentiment and Subsequent China Strategy
═══════════════════════════════════════════════════════════════════════════════════════════════════════
                                          Ordered Logit                                    OLS
                              ─────────────────────────────────────────────────    ─────────────
                                (1)        (2)        (3)        (4)        (5)        (6)
───────────────────────────────────────────────────────────────────────────────────────────────────────
Executive China Sentiment         0.544      0.506      0.537      0.527      0.523      0.237
                             (    2.40) (    2.19) (    2.31) (    2.27) (    2.20) (    2.27)
p-value                           0.016      0.028      0.021      0.023      0.028      0.023

Firm Controls               

# 4- Robust

In [28]:
import pandas as pd
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings
warnings.filterwarnings('ignore')


# Prepare Variables
dv = 'strategy_numeric'
iv = 'sentiment_mean_lag1'

# Controls (excluding macro for consistency)
controls = ['size_lag1', 'leverage_lag1', 'roa_lag1', 'cash_ratio_lag1',
            'ind_china_growth_lag1', 'lerner_index_lag1']

# Analysis variables
analysis_vars = [dv, iv] + controls + ['year', 'ticker']
df_analysis = df[analysis_vars].dropna().copy()

# Convert to numeric
df_analysis[dv] = pd.to_numeric(df_analysis[dv], errors='coerce')
df_analysis[iv] = pd.to_numeric(df_analysis[iv], errors='coerce')
for c in controls:
    df_analysis[c] = pd.to_numeric(df_analysis[c], errors='coerce')
df_analysis = df_analysis.dropna()

# Create ordered categorical DV
df_analysis['strategy_ordered'] = pd.Categorical(
    df_analysis[dv].astype(int),
    categories=[-1, 0, 1],
    ordered=True
)

print("=" * 80)
print("PANEL B: SUBSAMPLE ANALYSIS - TIME PERIOD")
print("=" * 80)


# Define Time Period Subsamples
subsamples = {
    'Pre-Trade War (2014-2017)': (2014, 2017),
    'Post-Trade War (2018-2025)': (2018, 2025),
    'Pre-COVID (2014-2019)': (2014, 2019),
    'Post-COVID (2020-2025)': (2020, 2025)
}

# Store results
results_dict = {}


# Run Ordered Logit for Each Subsample
for subsample_name, (year_start, year_end) in subsamples.items():
    print(f"\n{'='*70}")
    print(f"Subsample: {subsample_name}")
    print(f"{'='*70}")

    # Filter data
    df_sub = df_analysis[(df_analysis['year'] >= year_start) &
                         (df_analysis['year'] <= year_end)].copy()

    print(f"Observations: {len(df_sub)}")
    print(f"Firms: {df_sub['ticker'].nunique()}")
    print(f"Year range: {df_sub['year'].min()} - {df_sub['year'].max()}")
    print(f"\nDV Distribution:")
    print(df_sub[dv].value_counts().sort_index())

    # Check if we have enough observations in each category
    dv_counts = df_sub[dv].value_counts()
    if len(dv_counts) < 3 or dv_counts.min() < 5:
        print(f"\nWARNING: Insufficient variation in DV for this subsample")
        print("Skipping ordered logit, trying OLS instead...")

        # Run OLS as fallback
        import statsmodels.api as sm
        X_sub = sm.add_constant(df_sub[[iv] + controls].astype(float))
        y_sub = df_sub[dv].astype(float)
        model_sub = sm.OLS(y_sub, X_sub).fit(cov_type='cluster',
                                              cov_kwds={'groups': df_sub['ticker']})

        results_dict[subsample_name] = {
            'coef': model_sub.params[iv],
            'se': model_sub.bse[iv],
            'z': model_sub.tvalues[iv],
            'p': model_sub.pvalues[iv],
            'n': len(df_sub),
            'model': 'OLS'
        }
        print(f"\nOLS Results:")
        print(f"  Coefficient: {model_sub.params[iv]:.4f}")
        print(f"  t-statistic: {model_sub.tvalues[iv]:.3f}")
        print(f"  p-value: {model_sub.pvalues[iv]:.4f}")
        continue

    # Prepare X and y
    X_sub = df_sub[[iv] + controls].astype(float)
    y_sub = df_sub['strategy_ordered']

    # Run Ordered Logit
    try:
        model_sub = OrderedModel(y_sub, X_sub, distr='logit')
        result_sub = model_sub.fit(method='bfgs', disp=False)

        results_dict[subsample_name] = {
            'coef': result_sub.params[iv],
            'se': result_sub.bse[iv],
            'z': result_sub.tvalues[iv],
            'p': result_sub.pvalues[iv],
            'n': len(df_sub),
            'pseudo_r2': result_sub.prsquared,
            'model': 'Ordered Logit'
        }

        print(f"\nOrdered Logit Results:")
        print(f"  Coefficient: {result_sub.params[iv]:.4f}")
        print(f"  z-statistic: {result_sub.tvalues[iv]:.3f}")
        print(f"  p-value: {result_sub.pvalues[iv]:.4f}")
        print(f"  Odds Ratio: {np.exp(result_sub.params[iv]):.3f}")
        print(f"  Pseudo R²: {result_sub.prsquared:.4f}")

    except Exception as e:
        print(f"\nERROR fitting model: {e}")
        results_dict[subsample_name] = None

PANEL B: SUBSAMPLE ANALYSIS - TIME PERIOD

Subsample: Pre-Trade War (2014-2017)
Observations: 132
Firms: 44
Year range: 2015 - 2017

DV Distribution:
strategy_numeric
-1    33
 0    53
 1    46
Name: count, dtype: int64

Ordered Logit Results:
  Coefficient: 0.2503
  z-statistic: 0.502
  p-value: 0.6157
  Odds Ratio: 1.284
  Pseudo R²: 0.0491

Subsample: Post-Trade War (2018-2025)
Observations: 356
Firms: 45
Year range: 2018 - 2025

DV Distribution:
strategy_numeric
-1    131
 0     94
 1    131
Name: count, dtype: int64

Ordered Logit Results:
  Coefficient: 0.5149
  z-statistic: 1.918
  p-value: 0.0551
  Odds Ratio: 1.673
  Pseudo R²: 0.0084

Subsample: Pre-COVID (2014-2019)
Observations: 220
Firms: 44
Year range: 2015 - 2019

DV Distribution:
strategy_numeric
-1    63
 0    74
 1    83
Name: count, dtype: int64

Ordered Logit Results:
  Coefficient: 0.4583
  z-statistic: 1.216
  p-value: 0.2241
  Odds Ratio: 1.581
  Pseudo R²: 0.0311

Subsample: Post-COVID (2020-2025)
Observations: 

In [29]:
# Summary Table
print("\n" + "=" * 80)
print("PANEL B: SUBSAMPLE ANALYSIS - TIME PERIOD (SUMMARY TABLE)")
print("=" * 80)

# Helper function for significance stars
def sig_stars(p):
    if p < 0.01:
        return '***'
    elif p < 0.05:
        return '**'
    elif p < 0.10:
        return '*'
    else:
        return ''

print(f"""
══════════════════════════════════════════════════════════════════════════════════════════════
Panel B: Subsample Analysis by Time Period
══════════════════════════════════════════════════════════════════════════════════════════════
                                    Trade War Split              COVID Split
                              ─────────────────────────    ─────────────────────────
                                Pre-TW       Post-TW         Pre-COVID    Post-COVID
                              (2014-2017)  (2018-2025)      (2014-2019)  (2020-2025)
                                  (1)          (2)              (3)          (4)
──────────────────────────────────────────────────────────────────────────────────────────────""")

# Extract results
labels = ['Pre-Trade War (2014-2017)', 'Post-Trade War (2018-2025)',
          'Pre-COVID (2014-2019)', 'Post-COVID (2020-2025)']

coefs = []
z_stats = []
p_vals = []
n_obs = []

for label in labels:
    if results_dict.get(label):
        coefs.append(f"{results_dict[label]['coef']:.3f}{sig_stars(results_dict[label]['p'])}")
        z_stats.append(f"({results_dict[label]['z']:.2f})")
        p_vals.append(f"{results_dict[label]['p']:.3f}")
        n_obs.append(f"{results_dict[label]['n']}")
    else:
        coefs.append("--")
        z_stats.append("--")
        p_vals.append("--")
        n_obs.append("--")

print(f"""
Executive China Sentiment    {coefs[0]:>12} {coefs[1]:>12}      {coefs[2]:>12} {coefs[3]:>12}
                             {z_stats[0]:>12} {z_stats[1]:>12}      {z_stats[2]:>12} {z_stats[3]:>12}

Firm Controls                        Yes          Yes              Yes          Yes
Industry Controls                    Yes          Yes              Yes          Yes

Observations                 {n_obs[0]:>12} {n_obs[1]:>12}      {n_obs[2]:>12} {n_obs[3]:>12}
──────────────────────────────────────────────────────────────────────────────────────────────
Notes: Ordered Logit estimates. DV: Strategy coded as -1 (Retrenchment), 0 (Maintain),
+1 (Double Down). All independent variables lagged by one year. z-statistics in parentheses.
*** p<0.01, ** p<0.05, * p<0.10.
══════════════════════════════════════════════════════════════════════════════════════════════
""")

# ============================================
# Interpretation
# ============================================
print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)

# Compare coefficients
pre_tw = results_dict.get('Pre-Trade War (2014-2017)')
post_tw = results_dict.get('Post-Trade War (2018-2025)')
pre_covid = results_dict.get('Pre-COVID (2014-2019)')
post_covid = results_dict.get('Post-COVID (2020-2025)')

print("""
Trade War Split Analysis:""")
if pre_tw and post_tw:
    print(f"  - Pre-Trade War coefficient:  {pre_tw['coef']:.3f} (p={pre_tw['p']:.3f})")
    print(f"  - Post-Trade War coefficient: {post_tw['coef']:.3f} (p={post_tw['p']:.3f})")
    if post_tw['coef'] > pre_tw['coef']:
        print(f"  → The sentiment-strategy relationship appears STRONGER after the trade war began.")
    else:
        print(f"  → The sentiment-strategy relationship appears WEAKER after the trade war began.")

print("""
COVID Split Analysis:""")
if pre_covid and post_covid:
    print(f"  - Pre-COVID coefficient:  {pre_covid['coef']:.3f} (p={pre_covid['p']:.3f})")
    print(f"  - Post-COVID coefficient: {post_covid['coef']:.3f} (p={post_covid['p']:.3f})")
    if post_covid['coef'] > pre_covid['coef']:
        print(f"  → The sentiment-strategy relationship appears STRONGER in the post-COVID period.")
    else:
        print(f"  → The sentiment-strategy relationship appears WEAKER in the post-COVID period.")




PANEL B: SUBSAMPLE ANALYSIS - TIME PERIOD (SUMMARY TABLE)

══════════════════════════════════════════════════════════════════════════════════════════════
Panel B: Subsample Analysis by Time Period
══════════════════════════════════════════════════════════════════════════════════════════════
                                    Trade War Split              COVID Split
                              ─────────────────────────    ─────────────────────────
                                Pre-TW       Post-TW         Pre-COVID    Post-COVID
                              (2014-2017)  (2018-2025)      (2014-2019)  (2020-2025)
                                  (1)          (2)              (3)          (4)
──────────────────────────────────────────────────────────────────────────────────────────────

Executive China Sentiment           0.250       0.515*             0.458        0.392
                                   (0.50)       (1.92)            (1.22)       (1.29)

Firm Controls            

In [30]:
# ============================================
# Optional: Formal Test of Coefficient Difference (Chow-like test)
# ============================================
print("\n" + "=" * 80)
print("FORMAL TEST: INTERACTION MODEL")
print("=" * 80)

# Create period indicators
df_analysis['post_trade_war'] = (df_analysis['year'] >= 2018).astype(int)
df_analysis['post_covid'] = (df_analysis['year'] >= 2020).astype(int)

# Test 1: Trade War Interaction
print("\nTest 1: Trade War Period Interaction")
print("-" * 50)

df_analysis['sentiment_x_post_tw'] = df_analysis[iv] * df_analysis['post_trade_war']

X_interact_tw = df_analysis[[iv, 'post_trade_war', 'sentiment_x_post_tw'] + controls].astype(float)
y_interact = df_analysis['strategy_ordered']

try:
    model_interact_tw = OrderedModel(y_interact, X_interact_tw, distr='logit')
    result_interact_tw = model_interact_tw.fit(method='bfgs', disp=False)

    print(f"  Sentiment (baseline):           {result_interact_tw.params[iv]:.3f} "
          f"(z={result_interact_tw.tvalues[iv]:.2f}, p={result_interact_tw.pvalues[iv]:.3f})")
    print(f"  Post-Trade War dummy:           {result_interact_tw.params['post_trade_war']:.3f} "
          f"(z={result_interact_tw.tvalues['post_trade_war']:.2f}, p={result_interact_tw.pvalues['post_trade_war']:.3f})")
    print(f"  Sentiment × Post-Trade War:     {result_interact_tw.params['sentiment_x_post_tw']:.3f} "
          f"(z={result_interact_tw.tvalues['sentiment_x_post_tw']:.2f}, p={result_interact_tw.pvalues['sentiment_x_post_tw']:.3f})")

    if result_interact_tw.pvalues['sentiment_x_post_tw'] < 0.10:
        print(f"\n  → The interaction term is SIGNIFICANT, suggesting the effect of sentiment")
        print(f"    on strategy differs between pre- and post-trade war periods.")
    else:
        print(f"\n  → The interaction term is NOT significant, suggesting the effect of sentiment")
        print(f"    on strategy is STABLE across trade war periods.")
except Exception as e:
    print(f"  Error: {e}")

# Test 2: COVID Interaction
print("\nTest 2: COVID Period Interaction")
print("-" * 50)

df_analysis['sentiment_x_post_covid'] = df_analysis[iv] * df_analysis['post_covid']

X_interact_covid = df_analysis[[iv, 'post_covid', 'sentiment_x_post_covid'] + controls].astype(float)

try:
    model_interact_covid = OrderedModel(y_interact, X_interact_covid, distr='logit')
    result_interact_covid = model_interact_covid.fit(method='bfgs', disp=False)

    print(f"  Sentiment (baseline):           {result_interact_covid.params[iv]:.3f} "
          f"(z={result_interact_covid.tvalues[iv]:.2f}, p={result_interact_covid.pvalues[iv]:.3f})")
    print(f"  Post-COVID dummy:               {result_interact_covid.params['post_covid']:.3f} "
          f"(z={result_interact_covid.tvalues['post_covid']:.2f}, p={result_interact_covid.pvalues['post_covid']:.3f})")
    print(f"  Sentiment × Post-COVID:         {result_interact_covid.params['sentiment_x_post_covid']:.3f} "
          f"(z={result_interact_covid.tvalues['sentiment_x_post_covid']:.2f}, p={result_interact_covid.pvalues['sentiment_x_post_covid']:.3f})")

    if result_interact_covid.pvalues['sentiment_x_post_covid'] < 0.10:
        print(f"\n  → The interaction term is SIGNIFICANT, suggesting the effect of sentiment")
        print(f"    on strategy differs between pre- and post-COVID periods.")
    else:
        print(f"\n  → The interaction term is NOT significant, suggesting the effect of sentiment")
        print(f"    on strategy is STABLE across COVID periods.")
except Exception as e:
    print(f"  Error: {e}")

print("\n" + "=" * 80)
print("END OF PANEL B ANALYSIS")
print("=" * 80)


FORMAL TEST: INTERACTION MODEL

Test 1: Trade War Period Interaction
--------------------------------------------------
  Sentiment (baseline):           0.395 (z=0.87, p=0.387)
  Post-Trade War dummy:           -0.203 (z=-0.89, p=0.376)
  Sentiment × Post-Trade War:     0.173 (z=0.33, p=0.744)

  → The interaction term is NOT significant, suggesting the effect of sentiment
    on strategy is STABLE across trade war periods.

Test 2: COVID Period Interaction
--------------------------------------------------
  Sentiment (baseline):           0.513 (z=1.42, p=0.156)
  Post-COVID dummy:               -0.239 (z=-1.15, p=0.252)
  Sentiment × Post-COVID:         -0.018 (z=-0.04, p=0.970)

  → The interaction term is NOT significant, suggesting the effect of sentiment
    on strategy is STABLE across COVID periods.

END OF PANEL B ANALYSIS


In [31]:
import pandas as pd
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Prepare Variables
dv = 'strategy_numeric'
iv = 'sentiment_mean_lag1'

# Controls (excluding macro for consistency)
controls = ['size_lag1', 'leverage_lag1', 'roa_lag1', 'cash_ratio_lag1',
            'ind_china_growth_lag1', 'lerner_index_lag1']

# Analysis variables
analysis_vars = [dv, iv] + controls + ['year', 'ticker', 'industry']
df_analysis = df[analysis_vars].dropna().copy()

# Convert to numeric
df_analysis[dv] = pd.to_numeric(df_analysis[dv], errors='coerce')
df_analysis[iv] = pd.to_numeric(df_analysis[iv], errors='coerce')
for c in controls:
    df_analysis[c] = pd.to_numeric(df_analysis[c], errors='coerce')
df_analysis = df_analysis.dropna()

# Create ordered categorical DV
df_analysis['strategy_ordered'] = pd.Categorical(
    df_analysis[dv].astype(int),
    categories=[-1, 0, 1],
    ordered=True
)

print("=" * 80)
print("PANEL C: SUBSAMPLE ANALYSIS - INDUSTRY")
print("=" * 80)

# Check Industry Distribution
print("\nIndustry Distribution in Full Sample:")
print(df_analysis['industry'].value_counts())
print(f"\nTotal observations: {len(df_analysis)}")


# Define Industry Subsamples
industries = {
    'IT': 'IT',
    'Pharma': 'Pharma',
    'Auto': 'Auto'
}

# Store results
results_dict = {}

# Helper function for significance stars
def sig_stars(p):
    if p < 0.01:
        return '***'
    elif p < 0.05:
        return '**'
    elif p < 0.10:
        return '*'
    else:
        return ''

# ============================================
# Run Ordered Logit for Each Industry
# ============================================
for industry_label, industry_code in industries.items():
    print(f"\n{'='*70}")
    print(f"Industry: {industry_label}")
    print(f"{'='*70}")

    # Filter data
    df_sub = df_analysis[df_analysis['industry'] == industry_code].copy()

    print(f"Observations: {len(df_sub)}")
    print(f"Firms: {df_sub['ticker'].nunique()}")
    print(f"Year range: {df_sub['year'].min()} - {df_sub['year'].max()}")
    print(f"\nDV Distribution:")
    print(df_sub[dv].value_counts().sort_index())

    # Check if we have enough observations in each category
    dv_counts = df_sub[dv].value_counts()
    min_count = dv_counts.min() if len(dv_counts) == 3 else 0

    if len(dv_counts) < 3 or min_count < 5:
        print(f"\nWARNING: Insufficient variation in DV for ordered logit (min category count: {min_count})")
        print("Running OLS as fallback...")

        # Run OLS as fallback
        X_sub = sm.add_constant(df_sub[[iv] + controls].astype(float))
        y_sub = df_sub[dv].astype(float)

        try:
            model_sub = sm.OLS(y_sub, X_sub).fit(cov_type='cluster',
                                                  cov_kwds={'groups': df_sub['ticker']})

            results_dict[industry_label] = {
                'coef': model_sub.params[iv],
                'se': model_sub.bse[iv],
                'z': model_sub.tvalues[iv],
                'p': model_sub.pvalues[iv],
                'n': len(df_sub),
                'n_firms': df_sub['ticker'].nunique(),
                'r2': model_sub.rsquared,
                'model': 'OLS'
            }
            print(f"\nOLS Results:")
            print(f"  Coefficient: {model_sub.params[iv]:.4f}")
            print(f"  t-statistic: {model_sub.tvalues[iv]:.3f}")
            print(f"  p-value: {model_sub.pvalues[iv]:.4f}")
        except Exception as e:
            print(f"  OLS also failed: {e}")
            results_dict[industry_label] = None
        continue

    # Prepare X and y for Ordered Logit
    # Remove industry controls since we're within industry
    controls_no_ind = ['size_lag1', 'leverage_lag1', 'roa_lag1', 'cash_ratio_lag1']

    X_sub = df_sub[[iv] + controls_no_ind].astype(float)
    y_sub = df_sub['strategy_ordered']

    # Run Ordered Logit
    try:
        model_sub = OrderedModel(y_sub, X_sub, distr='logit')
        result_sub = model_sub.fit(method='bfgs', disp=False)

        results_dict[industry_label] = {
            'coef': result_sub.params[iv],
            'se': result_sub.bse[iv],
            'z': result_sub.tvalues[iv],
            'p': result_sub.pvalues[iv],
            'n': len(df_sub),
            'n_firms': df_sub['ticker'].nunique(),
            'pseudo_r2': result_sub.prsquared,
            'model': 'Ordered Logit'
        }

        print(f"\nOrdered Logit Results:")
        print(f"  Coefficient: {result_sub.params[iv]:.4f}")
        print(f"  z-statistic: {result_sub.tvalues[iv]:.3f}")
        print(f"  p-value: {result_sub.pvalues[iv]:.4f}")
        print(f"  Odds Ratio: {np.exp(result_sub.params[iv]):.3f}")
        print(f"  Pseudo R²: {result_sub.prsquared:.4f}")

    except Exception as e:
        print(f"\nERROR fitting Ordered Logit: {e}")
        print("Trying OLS as fallback...")

        # Fallback to OLS
        try:
            X_sub_ols = sm.add_constant(df_sub[[iv] + controls_no_ind].astype(float))
            y_sub_ols = df_sub[dv].astype(float)
            model_sub = sm.OLS(y_sub_ols, X_sub_ols).fit(cov_type='cluster',
                                                          cov_kwds={'groups': df_sub['ticker']})

            results_dict[industry_label] = {
                'coef': model_sub.params[iv],
                'se': model_sub.bse[iv],
                'z': model_sub.tvalues[iv],
                'p': model_sub.pvalues[iv],
                'n': len(df_sub),
                'n_firms': df_sub['ticker'].nunique(),
                'r2': model_sub.rsquared,
                'model': 'OLS'
            }
            print(f"\nOLS Results (fallback):")
            print(f"  Coefficient: {model_sub.params[iv]:.4f}")
            print(f"  t-statistic: {model_sub.tvalues[iv]:.3f}")
            print(f"  p-value: {model_sub.pvalues[iv]:.4f}")
        except Exception as e2:
            print(f"  OLS also failed: {e2}")
            results_dict[industry_label] = None

# ============================================
# Summary Table
# ============================================
print("\n" + "=" * 80)
print("PANEL C: SUBSAMPLE ANALYSIS - INDUSTRY (SUMMARY TABLE)")
print("=" * 80)

print(f"""
══════════════════════════════════════════════════════════════════════════════════════════════
Panel C: Subsample Analysis by Industry
══════════════════════════════════════════════════════════════════════════════════════════════
                                           IT           Pharma          Auto
                                          (1)            (2)            (3)
──────────────────────────────────────────────────────────────────────────────────────────────""")

# Extract results
industry_labels = ['IT', 'Pharma', 'Auto']

coefs = []
z_stats = []
p_vals = []
n_obs = []
n_firms = []
model_types = []

for label in industry_labels:
    if results_dict.get(label):
        r = results_dict[label]
        coefs.append(f"{r['coef']:.3f}{sig_stars(r['p'])}")
        z_stats.append(f"({r['z']:.2f})")
        p_vals.append(f"{r['p']:.3f}")
        n_obs.append(f"{r['n']}")
        n_firms.append(f"{r['n_firms']}")
        model_types.append(r['model'])
    else:
        coefs.append("--")
        z_stats.append("--")
        p_vals.append("--")
        n_obs.append("--")
        n_firms.append("--")
        model_types.append("--")

print(f"""
Executive China Sentiment      {coefs[0]:>14} {coefs[1]:>14} {coefs[2]:>14}
                               {z_stats[0]:>14} {z_stats[1]:>14} {z_stats[2]:>14}

Firm Controls                           Yes            Yes            Yes

Observations                   {n_obs[0]:>14} {n_obs[1]:>14} {n_obs[2]:>14}
Number of Firms                {n_firms[0]:>14} {n_firms[1]:>14} {n_firms[2]:>14}
Model                          {model_types[0]:>14} {model_types[1]:>14} {model_types[2]:>14}
──────────────────────────────────────────────────────────────────────────────────────────────
Notes: Ordered Logit estimates (OLS if ordered logit fails to converge due to small sample).
DV: Strategy coded as -1 (Retrenchment), 0 (Maintain), +1 (Double Down).
All independent variables lagged by one year. Industry-level controls excluded in
within-industry regressions. z/t-statistics in parentheses.
*** p<0.01, ** p<0.05, * p<0.10.
══════════════════════════════════════════════════════════════════════════════════════════════
""")



PANEL C: SUBSAMPLE ANALYSIS - INDUSTRY

Industry Distribution in Full Sample:
industry
IT        231
Pharma    154
Auto      103
Name: count, dtype: int64

Total observations: 488

Industry: IT
Observations: 231
Firms: 21
Year range: 2015 - 2025

DV Distribution:
strategy_numeric
-1    61
 0    94
 1    76
Name: count, dtype: int64

Ordered Logit Results:
  Coefficient: 0.6246
  z-statistic: 1.868
  p-value: 0.0617
  Odds Ratio: 1.867
  Pseudo R²: 0.0137

Industry: Pharma
Observations: 154
Firms: 14
Year range: 2015 - 2025

DV Distribution:
strategy_numeric
-1    60
 0    33
 1    61
Name: count, dtype: int64

Ordered Logit Results:
  Coefficient: 0.2532
  z-statistic: 0.568
  p-value: 0.5703
  Odds Ratio: 1.288
  Pseudo R²: 0.0718

Industry: Auto
Observations: 103
Firms: 10
Year range: 2015 - 2025

DV Distribution:
strategy_numeric
-1    43
 0    20
 1    40
Name: count, dtype: int64

Ordered Logit Results:
  Coefficient: 0.8964
  z-statistic: 1.284
  p-value: 0.1992
  Odds Ratio: 2.4

In [32]:
# ============================================
# Interpretation
# ============================================
print("\n" + "=" * 80)
print("INTERPRETATION BY INDUSTRY")
print("=" * 80)

for label in industry_labels:
    r = results_dict.get(label)
    if r:
        sig = "significant" if r['p'] < 0.10 else "not significant"
        direction = "positive" if r['coef'] > 0 else "negative"
        print(f"\n{label} Industry:")
        print(f"  - Coefficient: {r['coef']:.3f} ({direction})")
        print(f"  - Statistical significance: {sig} (p = {r['p']:.3f})")
        if r['coef'] > 0 and r['p'] < 0.10:
            print(f"  → Executive sentiment POSITIVELY predicts China strategy in {label} sector.")
        elif r['coef'] < 0 and r['p'] < 0.10:
            print(f"  → Executive sentiment NEGATIVELY predicts China strategy in {label} sector.")
        else:
            print(f"  → No significant relationship detected in {label} sector.")

# ============================================
# Formal Test: Industry Interaction Model
# ============================================
print("\n" + "=" * 80)
print("FORMAL TEST: INDUSTRY INTERACTION MODEL")
print("=" * 80)

# Create industry dummies (IT as baseline)
df_analysis['ind_pharma'] = (df_analysis['industry'] == 'Pharma').astype(int)
df_analysis['ind_auto'] = (df_analysis['industry'] == 'Auto').astype(int)

# Create interaction terms
df_analysis['sentiment_x_pharma'] = df_analysis[iv] * df_analysis['ind_pharma']
df_analysis['sentiment_x_auto'] = df_analysis[iv] * df_analysis['ind_auto']

# Full interaction model
X_interact = df_analysis[[iv, 'ind_pharma', 'ind_auto',
                          'sentiment_x_pharma', 'sentiment_x_auto'] + controls].astype(float)
y_interact = df_analysis['strategy_ordered']

print("\nInteraction Model (IT as baseline):")
print("-" * 50)

try:
    model_interact = OrderedModel(y_interact, X_interact, distr='logit')
    result_interact = model_interact.fit(method='bfgs', disp=False)

    print(f"  Sentiment (IT baseline):        {result_interact.params[iv]:.3f} "
          f"(z={result_interact.tvalues[iv]:.2f}, p={result_interact.pvalues[iv]:.3f}){sig_stars(result_interact.pvalues[iv])}")
    print(f"  Pharma dummy:                   {result_interact.params['ind_pharma']:.3f} "
          f"(z={result_interact.tvalues['ind_pharma']:.2f}, p={result_interact.pvalues['ind_pharma']:.3f}){sig_stars(result_interact.pvalues['ind_pharma'])}")
    print(f"  Auto dummy:                     {result_interact.params['ind_auto']:.3f} "
          f"(z={result_interact.tvalues['ind_auto']:.2f}, p={result_interact.pvalues['ind_auto']:.3f}){sig_stars(result_interact.pvalues['ind_auto'])}")
    print(f"  Sentiment × Pharma:             {result_interact.params['sentiment_x_pharma']:.3f} "
          f"(z={result_interact.tvalues['sentiment_x_pharma']:.2f}, p={result_interact.pvalues['sentiment_x_pharma']:.3f}){sig_stars(result_interact.pvalues['sentiment_x_pharma'])}")
    print(f"  Sentiment × Auto:               {result_interact.params['sentiment_x_auto']:.3f} "
          f"(z={result_interact.tvalues['sentiment_x_auto']:.2f}, p={result_interact.pvalues['sentiment_x_auto']:.3f}){sig_stars(result_interact.pvalues['sentiment_x_auto'])}")

    # Joint test for interaction terms
    print(f"\nInterpretation:")
    print(f"  - Effect in IT:     {result_interact.params[iv]:.3f}")
    print(f"  - Effect in Pharma: {result_interact.params[iv] + result_interact.params['sentiment_x_pharma']:.3f}")
    print(f"  - Effect in Auto:   {result_interact.params[iv] + result_interact.params['sentiment_x_auto']:.3f}")

    # Check significance of interactions
    pharma_sig = result_interact.pvalues['sentiment_x_pharma'] < 0.10
    auto_sig = result_interact.pvalues['sentiment_x_auto'] < 0.10

    if pharma_sig or auto_sig:
        print(f"\n  → Significant industry heterogeneity detected:")
        if pharma_sig:
            print(f"    - Pharma differs from IT (p = {result_interact.pvalues['sentiment_x_pharma']:.3f})")
        if auto_sig:
            print(f"    - Auto differs from IT (p = {result_interact.pvalues['sentiment_x_auto']:.3f})")
    else:
        print(f"\n  → No significant industry heterogeneity detected.")
        print(f"    The sentiment-strategy relationship is STABLE across industries.")

except Exception as e:
    print(f"  Error fitting interaction model: {e}")

    # Fallback to OLS
    print("\n  Trying OLS interaction model...")
    try:
        X_interact_ols = sm.add_constant(X_interact)
        y_interact_ols = df_analysis[dv].astype(float)
        model_interact_ols = sm.OLS(y_interact_ols, X_interact_ols).fit(
            cov_type='cluster', cov_kwds={'groups': df_analysis['ticker']})

        print(f"  Sentiment (IT baseline):        {model_interact_ols.params[iv]:.3f} "
              f"(t={model_interact_ols.tvalues[iv]:.2f}, p={model_interact_ols.pvalues[iv]:.3f})")
        print(f"  Sentiment × Pharma:             {model_interact_ols.params['sentiment_x_pharma']:.3f} "
              f"(t={model_interact_ols.tvalues['sentiment_x_pharma']:.2f}, p={model_interact_ols.pvalues['sentiment_x_pharma']:.3f})")
        print(f"  Sentiment × Auto:               {model_interact_ols.params['sentiment_x_auto']:.3f} "
              f"(t={model_interact_ols.tvalues['sentiment_x_auto']:.2f}, p={model_interact_ols.pvalues['sentiment_x_auto']:.3f})")
    except Exception as e2:
        print(f"  OLS also failed: {e2}")

# ============================================
# Industry Characteristics Summary
# ============================================
print("\n" + "=" * 80)
print("INDUSTRY CHARACTERISTICS SUMMARY")
print("=" * 80)

for ind in ['IT', 'Pharma', 'Auto']:
    df_ind = df_analysis[df_analysis['industry'] == ind]
    print(f"\n{ind}:")
    print(f"  Observations: {len(df_ind)}")
    print(f"  Firms: {df_ind['ticker'].nunique()}")
    print(f"  Mean sentiment: {df_ind[iv].mean():.3f}")
    print(f"  Strategy distribution:")
    strat_dist = df_ind[dv].value_counts(normalize=True).sort_index()
    for val, pct in strat_dist.items():
        label = {-1: 'Retrenchment', 0: 'Maintain', 1: 'Double Down'}[val]
        print(f"    {label}: {pct*100:.1f}%")

print("\n" + "=" * 80)
print("END OF PANEL C ANALYSIS")
print("=" * 80)


INTERPRETATION BY INDUSTRY

IT Industry:
  - Coefficient: 0.625 (positive)
  - Statistical significance: significant (p = 0.062)
  → Executive sentiment POSITIVELY predicts China strategy in IT sector.

Pharma Industry:
  - Coefficient: 0.253 (positive)
  - Statistical significance: not significant (p = 0.570)
  → No significant relationship detected in Pharma sector.

Auto Industry:
  - Coefficient: 0.896 (positive)
  - Statistical significance: not significant (p = 0.199)
  → No significant relationship detected in Auto sector.

FORMAL TEST: INDUSTRY INTERACTION MODEL

Interaction Model (IT as baseline):
--------------------------------------------------
  Sentiment (IT baseline):        0.617 (z=1.94, p=0.053)*
  Pharma dummy:                   -0.346 (z=-1.10, p=0.273)
  Auto dummy:                     -0.924 (z=-2.58, p=0.010)***
  Sentiment × Pharma:             -0.305 (z=-0.56, p=0.577)
  Sentiment × Auto:               0.692 (z=0.91, p=0.361)

Interpretation:
  - Effect in IT:

In [33]:
import pandas as pd
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Prepare Variables

dv = 'strategy_numeric'

# Controls
controls = ['size_lag1', 'leverage_lag1', 'roa_lag1', 'cash_ratio_lag1',
            'ind_china_growth_lag1', 'lerner_index_lag1']

# Check available sentiment measures
print("=" * 80)
print("PANEL D: MEASUREMENT ROBUSTNESS - ALTERNATIVE SENTIMENT MEASURES")
print("=" * 80)

print("\nAvailable sentiment-related columns:")
sent_cols = [c for c in df.columns if 'sent' in c.lower() or 'positive' in c.lower() or 'negative' in c.lower()]
print(sent_cols)

# ============================================
# Define Alternative Sentiment Measures
# ============================================

# Create lagged versions of alternative sentiment measures if not exist
if 'sentiment_positive_pct_lag1' not in df.columns:
    df['sentiment_positive_pct_lag1'] = df.groupby('ticker')['sentiment_positive_pct'].shift(1)

if 'sentiment_negative_pct_lag1' not in df.columns:
    df['sentiment_negative_pct_lag1'] = df.groupby('ticker')['sentiment_negative_pct'].shift(1)

# Create net sentiment (positive - negative)
df['sentiment_net'] = df['sentiment_positive_pct'] - df['sentiment_negative_pct']
df['sentiment_net_lag1'] = df.groupby('ticker')['sentiment_net'].shift(1)

# Create sentiment ratio (positive / (positive + negative)) - avoiding division by zero
df['sentiment_ratio'] = df['sentiment_positive_pct'] / (df['sentiment_positive_pct'] + df['sentiment_negative_pct'] + 0.001)
df['sentiment_ratio_lag1'] = df.groupby('ticker')['sentiment_ratio'].shift(1)

# Define alternative IV measures
alternative_ivs = {
    'Mean Sentiment (Baseline)': 'sentiment_mean_lag1',
    'Positive Tone Share': 'sentiment_positive_pct_lag1',
    'Negative Tone Share': 'sentiment_negative_pct_lag1',
    'Net Sentiment (Pos - Neg)': 'sentiment_net_lag1',
    'Sentiment Ratio (Pos/Total)': 'sentiment_ratio_lag1'
}

# ============================================
# Prepare Analysis Sample
# ============================================
analysis_vars = [dv] + list(alternative_ivs.values()) + controls + ['year', 'ticker']
analysis_vars = list(set(analysis_vars))  # Remove duplicates

df_analysis = df[analysis_vars].dropna().copy()

# Convert to numeric
df_analysis[dv] = pd.to_numeric(df_analysis[dv], errors='coerce')
for iv in alternative_ivs.values():
    df_analysis[iv] = pd.to_numeric(df_analysis[iv], errors='coerce')
for c in controls:
    df_analysis[c] = pd.to_numeric(df_analysis[c], errors='coerce')

df_analysis = df_analysis.dropna()

# Create ordered categorical DV
df_analysis['strategy_ordered'] = pd.Categorical(
    df_analysis[dv].astype(int),
    categories=[-1, 0, 1],
    ordered=True
)

print(f"\nAnalysis sample size: {len(df_analysis)}")

# ============================================
# Descriptive Statistics for Sentiment Measures
# ============================================
print("\n" + "=" * 80)
print("DESCRIPTIVE STATISTICS: ALTERNATIVE SENTIMENT MEASURES")
print("=" * 80)

print(f"\n{'Measure':<35} {'Mean':>10} {'SD':>10} {'Min':>10} {'Max':>10}")
print("-" * 75)
for name, var in alternative_ivs.items():
    stats = df_analysis[var].describe()
    print(f"{name:<35} {stats['mean']:>10.3f} {stats['std']:>10.3f} {stats['min']:>10.3f} {stats['max']:>10.3f}")

# Correlation matrix
print("\n" + "-" * 80)
print("CORRELATION MATRIX OF SENTIMENT MEASURES")
print("-" * 80)
corr_vars = list(alternative_ivs.values())
corr_matrix = df_analysis[corr_vars].corr()

# Rename for display
corr_display = corr_matrix.copy()
corr_display.index = ['Mean', 'Pos%', 'Neg%', 'Net', 'Ratio']
corr_display.columns = ['Mean', 'Pos%', 'Neg%', 'Net', 'Ratio']
print(corr_display.round(3))

# ============================================
# Helper Functions
# ============================================
def sig_stars(p):
    if p < 0.01:
        return '***'
    elif p < 0.05:
        return '**'
    elif p < 0.10:
        return '*'
    else:
        return ''

# ============================================
# Run Ordered Logit for Each Sentiment Measure
# ============================================
print("\n" + "=" * 80)
print("ORDERED LOGIT RESULTS: ALTERNATIVE SENTIMENT MEASURES")
print("=" * 80)

results_dict = {}

for measure_name, iv in alternative_ivs.items():
    print(f"\n{'-'*70}")
    print(f"Sentiment Measure: {measure_name}")
    print(f"Variable: {iv}")
    print(f"{'-'*70}")

    # Prepare X and y
    X = df_analysis[[iv] + controls].astype(float)
    y = df_analysis['strategy_ordered']

    # For negative tone, we expect opposite sign
    expected_sign = "negative" if "negative" in iv.lower() else "positive"

    try:
        model = OrderedModel(y, X, distr='logit')
        result = model.fit(method='bfgs', disp=False)

        coef = result.params[iv]
        se = result.bse[iv]
        z = result.tvalues[iv]
        p = result.pvalues[iv]

        results_dict[measure_name] = {
            'coef': coef,
            'se': se,
            'z': z,
            'p': p,
            'pseudo_r2': result.prsquared,
            'n': len(df_analysis),
            'expected_sign': expected_sign
        }

        print(f"  Coefficient: {coef:.4f} {sig_stars(p)}")
        print(f"  Std. Error:  {se:.4f}")
        print(f"  z-statistic: {z:.3f}")
        print(f"  p-value:     {p:.4f}")
        print(f"  Odds Ratio:  {np.exp(coef):.3f}")
        print(f"  Pseudo R²:   {result.prsquared:.4f}")
        print(f"  Expected sign: {expected_sign}")

        # Check if sign matches expectation
        if expected_sign == "positive" and coef > 0:
            print(f"  ✓ Sign matches expectation (positive sentiment → positive strategy)")
        elif expected_sign == "negative" and coef < 0:
            print(f"  ✓ Sign matches expectation (negative sentiment → negative strategy)")
        elif expected_sign == "negative" and coef > 0:
            print(f"  ✗ Sign opposite to expectation (but may indicate negative tone → retrenchment)")
        else:
            print(f"  ? Sign does not match simple expectation")

    except Exception as e:
        print(f"  ERROR: {e}")
        results_dict[measure_name] = None



PANEL D: MEASUREMENT ROBUSTNESS - ALTERNATIVE SENTIMENT MEASURES

Available sentiment-related columns:
['total_china_sentences', 'sentiment_mean', 'sentiment_positive_pct', 'sentiment_negative_pct', 'non_action_sentence_count', 'has_sentiment', 'sentiment_mean_lag1']

Analysis sample size: 488

DESCRIPTIVE STATISTICS: ALTERNATIVE SENTIMENT MEASURES

Measure                                   Mean         SD        Min        Max
---------------------------------------------------------------------------
Mean Sentiment (Baseline)                0.197      0.365     -1.000      1.000
Positive Tone Share                      0.360      0.261      0.000      1.000
Negative Tone Share                      0.163      0.193      0.000      1.000
Net Sentiment (Pos - Neg)                0.197      0.365     -1.000      1.000
Sentiment Ratio (Pos/Total)              0.620      0.346      0.000      0.999

--------------------------------------------------------------------------------
CORRELATIO

In [34]:
# ============================================
# Summary Table
# ============================================
print("\n" + "=" * 80)
print("PANEL D: MEASUREMENT ROBUSTNESS (SUMMARY TABLE)")
print("=" * 80)

print(f"""
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Panel D: Measurement Robustness - Alternative Sentiment Operationalizations
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
                                      Mean Sent.    Positive     Negative    Net Sent.    Sent. Ratio
                                      (Baseline)    Tone Share   Tone Share  (Pos-Neg)    (Pos/Total)
                                          (1)          (2)          (3)          (4)          (5)
──────────────────────────────────────────────────────────────────────────────────────────────────────────────""")

# Extract results
measures = ['Mean Sentiment (Baseline)', 'Positive Tone Share', 'Negative Tone Share',
            'Net Sentiment (Pos - Neg)', 'Sentiment Ratio (Pos/Total)']

coefs = []
z_stats = []
p_vals = []

for m in measures:
    if results_dict.get(m):
        r = results_dict[m]
        coefs.append(f"{r['coef']:.3f}{sig_stars(r['p'])}")
        z_stats.append(f"({r['z']:.2f})")
        p_vals.append(f"{r['p']:.3f}")
    else:
        coefs.append("--")
        z_stats.append("--")
        p_vals.append("--")

print(f"""
Sentiment Measure                 {coefs[0]:>12} {coefs[1]:>12} {coefs[2]:>12} {coefs[3]:>12} {coefs[4]:>12}
                                  {z_stats[0]:>12} {z_stats[1]:>12} {z_stats[2]:>12} {z_stats[3]:>12} {z_stats[4]:>12}

Expected Sign                          (+)          (+)          (-)          (+)          (+)

Firm Controls                          Yes          Yes          Yes          Yes          Yes
Industry Controls                      Yes          Yes          Yes          Yes          Yes

Observations                           {len(df_analysis):>3}          {len(df_analysis):>3}          {len(df_analysis):>3}          {len(df_analysis):>3}          {len(df_analysis):>3}
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
Notes: Ordered Logit estimates. DV: Strategy coded as -1 (Retrenchment), 0 (Maintain), +1 (Double Down).
All independent variables lagged by one year. z-statistics in parentheses.
Expected signs: Positive/Net/Ratio measures should have (+) coefficients; Negative tone should have (-).
*** p<0.01, ** p<0.05, * p<0.10.
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
""")

# ============================================
# Interpretation
# ============================================
print("\n" + "=" * 80)
print("INTERPRETATION: MEASUREMENT ROBUSTNESS")
print("=" * 80)

# Count how many measures show expected sign
expected_signs_match = 0
significant_count = 0

for m in measures:
    if results_dict.get(m):
        r = results_dict[m]
        if r['expected_sign'] == 'positive' and r['coef'] > 0:
            expected_signs_match += 1
        elif r['expected_sign'] == 'negative' and r['coef'] < 0:
            expected_signs_match += 1
        if r['p'] < 0.10:
            significant_count += 1

print(f"""
Summary:
  - {expected_signs_match} out of {len(measures)} measures show the expected sign direction
  - {significant_count} out of {len(measures)} measures are statistically significant (p < 0.10)
""")

# Detailed interpretation
print("Detailed Results:")
print("-" * 50)

for m in measures:
    if results_dict.get(m):
        r = results_dict[m]
        sig = "significant" if r['p'] < 0.10 else "not significant"
        direction = "positive" if r['coef'] > 0 else "negative"
        expected = r['expected_sign']
        match = "✓" if (expected == 'positive' and r['coef'] > 0) or (expected == 'negative' and r['coef'] < 0) else "✗"

        print(f"\n{m}:")
        print(f"  Coefficient: {r['coef']:.3f} ({direction}), p = {r['p']:.3f} ({sig})")
        print(f"  Expected: {expected}, Actual: {direction} {match}")

# ============================================
# Robustness Assessment
# ============================================
print("\n" + "=" * 80)
print("ROBUSTNESS ASSESSMENT")
print("=" * 80)

# Check consistency
baseline_coef = results_dict['Mean Sentiment (Baseline)']['coef']
baseline_sig = results_dict['Mean Sentiment (Baseline)']['p'] < 0.10

consistent_direction = sum(1 for m in measures if results_dict.get(m) and
                           ((results_dict[m]['expected_sign'] == 'positive' and results_dict[m]['coef'] > 0) or
                            (results_dict[m]['expected_sign'] == 'negative' and results_dict[m]['coef'] < 0)))

print(f"""
Baseline Result (Mean Sentiment):
  - Coefficient: {baseline_coef:.3f}
  - Significant: {'Yes' if baseline_sig else 'No'} (p = {results_dict['Mean Sentiment (Baseline)']['p']:.3f})

Robustness Check:
  - {consistent_direction}/{len(measures)} measures show expected sign direction
  - Main finding {'IS' if consistent_direction >= 3 else 'is NOT'} robust to alternative sentiment operationalizations
""")

if consistent_direction >= 4:
    print("""
Conclusion: STRONG robustness
  The positive relationship between executive China sentiment and subsequent
  China strategy is robust across multiple sentiment operationalizations.
  This suggests the finding is not driven by a specific measurement choice.
""")
elif consistent_direction >= 3:
    print("""
Conclusion: MODERATE robustness
  The main finding is generally supported by alternative measures, though
  some variation exists. The overall pattern is consistent with the baseline.
""")
else:
    print("""
Conclusion: WEAK robustness
  Results vary substantially across different sentiment measures.
  Interpretation should be cautious.
""")

# ============================================
# Additional: Standardized Coefficients for Comparison
# ============================================
print("\n" + "=" * 80)
print("STANDARDIZED COEFFICIENTS (for magnitude comparison)")
print("=" * 80)

print(f"\n{'Measure':<35} {'Raw Coef':>12} {'SD of IV':>12} {'Std. Coef':>12}")
print("-" * 75)

for m, iv in alternative_ivs.items():
    if results_dict.get(m):
        r = results_dict[m]
        sd_iv = df_analysis[iv].std()
        std_coef = r['coef'] * sd_iv  # Standardized coefficient
        print(f"{m:<35} {r['coef']:>12.3f} {sd_iv:>12.3f} {std_coef:>12.3f}")

print("""
Note: Standardized coefficient = Raw coefficient × SD of IV
      This allows comparison of effect magnitudes across different scales.
""")

print("\n" + "=" * 80)
print("END OF PANEL D ANALYSIS")
print("=" * 80)


PANEL D: MEASUREMENT ROBUSTNESS (SUMMARY TABLE)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Panel D: Measurement Robustness - Alternative Sentiment Operationalizations
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
                                      Mean Sent.    Positive     Negative    Net Sent.    Sent. Ratio
                                      (Baseline)    Tone Share   Tone Share  (Pos-Neg)    (Pos/Total)
                                          (1)          (2)          (3)          (4)          (5)
──────────────────────────────────────────────────────────────────────────────────────────────────────────────

Sentiment Measure                      0.537**        0.412    -1.148***      0.537**        0.317
                                        (2.31)       (1.26)      (-2.64)       (2.31)       (1.32)

Expected Sign                          

# 5-ENDOGENEITY CONCERNS

In [35]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# ============================================
# Prepare Variables
# ============================================
dv = 'strategy_numeric'
iv_lag = 'sentiment_mean_lag1'
iv_current = 'sentiment_mean'

controls = ['size_lag1', 'leverage_lag1', 'roa_lag1', 'cash_ratio_lag1',
            'ind_china_growth_lag1', 'lerner_index_lag1']

# Prepare analysis sample
analysis_vars = [dv, iv_lag, iv_current, 'year', 'ticker'] + controls
df_analysis = df[analysis_vars].copy()

# Convert to numeric
df_analysis[dv] = pd.to_numeric(df_analysis[dv], errors='coerce')
df_analysis[iv_lag] = pd.to_numeric(df_analysis[iv_lag], errors='coerce')
df_analysis[iv_current] = pd.to_numeric(df_analysis[iv_current], errors='coerce')
for c in controls:
    df_analysis[c] = pd.to_numeric(df_analysis[c], errors='coerce')

df_analysis = df_analysis.dropna()

# Create numeric firm ID
df_analysis['firm_id'] = pd.factorize(df_analysis['ticker'])[0]

print("=" * 80)
print("ADDRESSING ENDOGENEITY CONCERNS")
print("=" * 80)
print(f"Sample size: {len(df_analysis)}")
print(f"Number of firms: {df_analysis['firm_id'].nunique()}")

# Helper function
def sig_stars(p):
    if p < 0.01: return '***'
    elif p < 0.05: return '**'
    elif p < 0.10: return '*'
    else: return ''

# ============================================
# METHOD 1: Firm + Year Fixed Effects
# ============================================
print("\n" + "=" * 80)
print("METHOD 1: FIRM + YEAR FIXED EFFECTS")
print("=" * 80)

print("""
Logic: Controls for all time-invariant firm characteristics (CEO style,
corporate culture, unobserved China exposure, etc.)
""")

# Create dummies with clean names
firm_dummies = pd.get_dummies(df_analysis['firm_id'], prefix='f', drop_first=True).astype(float)
year_dummies = pd.get_dummies(df_analysis['year'], prefix='y', drop_first=True).astype(float)

# Reset all indices
df_reset = df_analysis.reset_index(drop=True)
firm_dummies = firm_dummies.reset_index(drop=True)
year_dummies = year_dummies.reset_index(drop=True)

# Build X matrix
X_vars = df_reset[[iv_lag] + controls].astype(float)
X_fe = pd.concat([X_vars, firm_dummies, year_dummies], axis=1)
X_fe = sm.add_constant(X_fe)
y_fe = df_reset[dv].astype(float)

# Cluster variable
clusters = df_reset['firm_id']

model_fe = sm.OLS(y_fe, X_fe).fit(cov_type='cluster', cov_kwds={'groups': clusters})

fe_coef = model_fe.params[iv_lag]
fe_t = model_fe.tvalues[iv_lag]
fe_p = model_fe.pvalues[iv_lag]

print(f"Sentiment (t-1) coefficient: {fe_coef:.4f} {sig_stars(fe_p)}")
print(f"t-statistic: {fe_t:.3f}")
print(f"p-value: {fe_p:.4f}")

# ============================================
# METHOD 2: First Difference Model
# ============================================
print("\n" + "=" * 80)
print("METHOD 2: FIRST DIFFERENCE MODEL (ΔSent → ΔStrategy)")
print("=" * 80)

print("""
Logic: Does CHANGE in sentiment predict CHANGE in strategy?
Removes all time-invariant confounders.
""")

df_sorted = df_analysis.sort_values(['ticker', 'year']).copy()

# Create differences
df_sorted['strategy_diff'] = df_sorted.groupby('ticker')[dv].diff()
df_sorted['sentiment_diff'] = df_sorted.groupby('ticker')[iv_lag].diff()

for c in controls:
    df_sorted[f'{c}_diff'] = df_sorted.groupby('ticker')[c].diff()

df_diff = df_sorted.dropna(subset=['strategy_diff', 'sentiment_diff']).reset_index(drop=True)

print(f"Observations after differencing: {len(df_diff)}")

control_diffs = [f'{c}_diff' for c in controls]
X_diff = sm.add_constant(df_diff[['sentiment_diff'] + control_diffs].astype(float))
y_diff = df_diff['strategy_diff'].astype(float)

model_diff = sm.OLS(y_diff, X_diff).fit(cov_type='cluster',
                                         cov_kwds={'groups': df_diff['ticker']})

diff_coef = model_diff.params['sentiment_diff']
diff_t = model_diff.tvalues['sentiment_diff']
diff_p = model_diff.pvalues['sentiment_diff']

print(f"ΔSentiment coefficient: {diff_coef:.4f} {sig_stars(diff_p)}")
print(f"t-statistic: {diff_t:.3f}")
print(f"p-value: {diff_p:.4f}")

# ============================================
# METHOD 3: Placebo Test (Future Sentiment)
# ============================================
print("\n" + "=" * 80)
print("METHOD 3: PLACEBO TEST (Future Sentiment t+1)")
print("=" * 80)

print("""
Logic: FUTURE sentiment should NOT predict CURRENT strategy.
If it does → omitted variable problem.
Expected: Coefficient should be INSIGNIFICANT.
""")

df_sorted['sentiment_lead1'] = df_sorted.groupby('ticker')[iv_current].shift(-1)
df_placebo = df_sorted.dropna(subset=['sentiment_lead1']).reset_index(drop=True)

print(f"Observations: {len(df_placebo)}")

X_placebo = sm.add_constant(df_placebo[['sentiment_lead1'] + controls].astype(float))
y_placebo = df_placebo[dv].astype(float)

model_placebo = sm.OLS(y_placebo, X_placebo).fit(cov_type='cluster',
                                                   cov_kwds={'groups': df_placebo['ticker']})

placebo_coef = model_placebo.params['sentiment_lead1']
placebo_t = model_placebo.tvalues['sentiment_lead1']
placebo_p = model_placebo.pvalues['sentiment_lead1']

print(f"Future Sentiment (t+1) coefficient: {placebo_coef:.4f} {sig_stars(placebo_p)}")
print(f"t-statistic: {placebo_t:.3f}")
print(f"p-value: {placebo_p:.4f}")

if placebo_p > 0.10:
    print("\n✓ PASSED: Future sentiment does NOT predict current strategy.")
else:
    print("\n✗ WARNING: Future sentiment predicts current strategy.")

# ============================================
# METHOD 4: Granger Causality Test
# ============================================
print("\n" + "=" * 80)
print("METHOD 4: GRANGER-STYLE CAUSALITY TEST")
print("=" * 80)

print("""
Logic: Test both directions:
- Sentiment(t-1) → Strategy(t): Expected SIGNIFICANT
- Strategy(t-1) → Sentiment(t): Expected NOT significant
""")

df_sorted['strategy_lag1'] = df_sorted.groupby('ticker')[dv].shift(1)
df_granger = df_sorted.dropna(subset=['strategy_lag1', iv_lag]).reset_index(drop=True)

print(f"Observations: {len(df_granger)}")

# Direction A: Sentiment → Strategy
print("\nTest A: Sentiment(t-1) → Strategy(t)")
X_g1 = sm.add_constant(df_granger[[iv_lag, 'strategy_lag1'] + controls].astype(float))
y_g1 = df_granger[dv].astype(float)

model_g1 = sm.OLS(y_g1, X_g1).fit(cov_type='cluster', cov_kwds={'groups': df_granger['ticker']})

g1_coef = model_g1.params[iv_lag]
g1_t = model_g1.tvalues[iv_lag]
g1_p = model_g1.pvalues[iv_lag]

print(f"  Sentiment(t-1): coef={g1_coef:.4f} {sig_stars(g1_p)}, t={g1_t:.2f}, p={g1_p:.3f}")

# Direction B: Strategy → Sentiment
print("\nTest B: Strategy(t-1) → Sentiment(t)")
X_g2 = sm.add_constant(df_granger[['strategy_lag1'] + controls].astype(float))
y_g2 = df_granger[iv_current].astype(float)

model_g2 = sm.OLS(y_g2, X_g2).fit(cov_type='cluster', cov_kwds={'groups': df_granger['ticker']})

g2_coef = model_g2.params['strategy_lag1']
g2_t = model_g2.tvalues['strategy_lag1']
g2_p = model_g2.pvalues['strategy_lag1']

print(f"  Strategy(t-1): coef={g2_coef:.4f} {sig_stars(g2_p)}, t={g2_t:.2f}, p={g2_p:.3f}")

print("\nInterpretation:")
if g1_p < 0.10 and g2_p > 0.10:
    print("  ✓ PASSED: Sentiment → Strategy, but NOT reverse.")
elif g1_p < 0.10 and g2_p < 0.10:
    print("  ⚠ Bidirectional: Both directions significant.")
else:
    print("  ? Inconclusive.")

# ============================================
# METHOD 5: Dynamic Panel
# ============================================
print("\n" + "=" * 80)
print("METHOD 5: DYNAMIC PANEL (+ Lagged Strategy)")
print("=" * 80)

print("""
Logic: Control for strategic inertia/persistence.
Does sentiment still matter after controlling for past strategy?
""")

# Same as Granger Test A
dynamic_coef = g1_coef
dynamic_t = g1_t
dynamic_p = g1_p

strat_lag_coef = model_g1.params['strategy_lag1']
strat_lag_t = model_g1.tvalues['strategy_lag1']
strat_lag_p = model_g1.pvalues['strategy_lag1']

print(f"Sentiment(t-1):  coef={dynamic_coef:.4f} {sig_stars(dynamic_p)}, t={dynamic_t:.2f}, p={dynamic_p:.3f}")
print(f"Strategy(t-1):   coef={strat_lag_coef:.4f} {sig_stars(strat_lag_p)}, t={strat_lag_t:.2f}, p={strat_lag_p:.3f}")



ADDRESSING ENDOGENEITY CONCERNS
Sample size: 488
Number of firms: 45

METHOD 1: FIRM + YEAR FIXED EFFECTS

Logic: Controls for all time-invariant firm characteristics (CEO style,
corporate culture, unobserved China exposure, etc.)

Sentiment (t-1) coefficient: 0.2073 **
t-statistic: 2.249
p-value: 0.0245

METHOD 2: FIRST DIFFERENCE MODEL (ΔSent → ΔStrategy)

Logic: Does CHANGE in sentiment predict CHANGE in strategy?
Removes all time-invariant confounders.

Observations after differencing: 443
ΔSentiment coefficient: 0.0196 
t-statistic: 0.199
p-value: 0.8422

METHOD 3: PLACEBO TEST (Future Sentiment t+1)

Logic: FUTURE sentiment should NOT predict CURRENT strategy.
If it does → omitted variable problem.
Expected: Coefficient should be INSIGNIFICANT.

Observations: 443
Future Sentiment (t+1) coefficient: 0.3380 ***
t-statistic: 2.730
p-value: 0.0063

✗ WARNING: Future sentiment predicts current strategy.

METHOD 4: GRANGER-STYLE CAUSALITY TEST

Logic: Test both directions:
- Sentiment(

In [36]:
# ============================================
# SUMMARY TABLE
# ============================================
print("\n" + "=" * 80)
print("SUMMARY TABLE: ADDRESSING ENDOGENEITY")
print("=" * 80)

print(f"""
══════════════════════════════════════════════════════════════════════════════════════════
Table: Addressing Endogeneity Concerns
══════════════════════════════════════════════════════════════════════════════════════════
                                                    Coef.        t-stat     p-value
──────────────────────────────────────────────────────────────────────────────────────────
Panel A: Omitted Variables

(1) Firm + Year Fixed Effects                      {fe_coef:>7.3f}{sig_stars(fe_p):<3}      {fe_t:>6.2f}      {fe_p:.3f}
(2) First Difference (ΔSent → ΔStrat)              {diff_coef:>7.3f}{sig_stars(diff_p):<3}      {diff_t:>6.2f}      {diff_p:.3f}

Panel B: Reverse Causality

(3) Placebo: Sentiment(t+1) → Strategy(t)          {placebo_coef:>7.3f}{sig_stars(placebo_p):<3}      {placebo_t:>6.2f}      {placebo_p:.3f}
    Expected: NOT significant                      {'✓ PASS' if placebo_p > 0.10 else '✗ FAIL'}

(4a) Granger: Sent(t-1) → Strat(t)                 {g1_coef:>7.3f}{sig_stars(g1_p):<3}      {g1_t:>6.2f}      {g1_p:.3f}
(4b) Granger: Strat(t-1) → Sent(t)                 {g2_coef:>7.3f}{sig_stars(g2_p):<3}      {g2_t:>6.2f}      {g2_p:.3f}
    Expected: (4a) sig, (4b) not sig               {'✓ PASS' if (g1_p < 0.10 and g2_p > 0.10) else '⚠ CHECK'}

(5) Dynamic Panel (+ Strategy(t-1))                {dynamic_coef:>7.3f}{sig_stars(dynamic_p):<3}      {dynamic_t:>6.2f}      {dynamic_p:.3f}

──────────────────────────────────────────────────────────────────────────────────────────
Notes: Standard errors clustered by firm. *** p<0.01, ** p<0.05, * p<0.10
══════════════════════════════════════════════════════════════════════════════════════════
""")



SUMMARY TABLE: ADDRESSING ENDOGENEITY

══════════════════════════════════════════════════════════════════════════════════════════
Table: Addressing Endogeneity Concerns
══════════════════════════════════════════════════════════════════════════════════════════
                                                    Coef.        t-stat     p-value
──────────────────────────────────────────────────────────────────────────────────────────
Panel A: Omitted Variables

(1) Firm + Year Fixed Effects                        0.207**         2.25      0.025
(2) First Difference (ΔSent → ΔStrat)                0.020           0.20      0.842

Panel B: Reverse Causality

(3) Placebo: Sentiment(t+1) → Strategy(t)            0.338***        2.73      0.006
    Expected: NOT significant                      ✗ FAIL

(4a) Granger: Sent(t-1) → Strat(t)                   0.115           1.41      0.159
(4b) Granger: Strat(t-1) → Sent(t)                   0.059***        2.63      0.008
    Expected: (4a) sig,

In [37]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')


# ============================================
# Prepare Variables
# ============================================
dv = 'strategy_numeric'
iv_lag = 'sentiment_mean_lag1'
iv_current = 'sentiment_mean'

controls = ['size_lag1', 'leverage_lag1', 'roa_lag1', 'cash_ratio_lag1',
            'ind_china_growth_lag1', 'lerner_index_lag1']

# Prepare analysis sample
analysis_vars = [dv, iv_lag, iv_current, 'year', 'ticker'] + controls
df_analysis = df[analysis_vars].copy()

# Convert to numeric
df_analysis[dv] = pd.to_numeric(df_analysis[dv], errors='coerce')
df_analysis[iv_lag] = pd.to_numeric(df_analysis[iv_lag], errors='coerce')
df_analysis[iv_current] = pd.to_numeric(df_analysis[iv_current], errors='coerce')
for c in controls:
    df_analysis[c] = pd.to_numeric(df_analysis[c], errors='coerce')

df_analysis = df_analysis.dropna()
df_analysis['firm_id'] = pd.factorize(df_analysis['ticker'])[0]

print("=" * 80)
print("ENDOGENEITY TESTS: PSM AND GMM")
print("=" * 80)
print(f"Sample size: {len(df_analysis)}")
print(f"Number of firms: {df_analysis['firm_id'].nunique()}")

# Helper function
def sig_stars(p):
    if p < 0.01: return '***'
    elif p < 0.05: return '**'
    elif p < 0.10: return '*'
    else: return ''

# ============================================
# METHOD 1: PROPENSITY SCORE MATCHING (PSM)
# ============================================
print("\n" + "=" * 80)
print("METHOD 1: PROPENSITY SCORE MATCHING (PSM)")
print("=" * 80)

print("""
Logic:
1. Divide firms into High vs Low sentiment groups
2. Match similar firms based on observable characteristics
3. Compare strategy outcomes between matched pairs
4. This controls for selection bias based on observables
""")

# Step 1: Create treatment variable (High Sentiment = 1, Low Sentiment = 0)
median_sentiment = df_analysis[iv_lag].median()
df_analysis['high_sentiment'] = (df_analysis[iv_lag] > median_sentiment).astype(int)

print(f"Median sentiment: {median_sentiment:.4f}")
print(f"High sentiment group: {df_analysis['high_sentiment'].sum()} obs")
print(f"Low sentiment group: {(1 - df_analysis['high_sentiment']).sum()} obs")

# Step 2: Estimate propensity score (probability of high sentiment)
print("\n--- Step 1: Estimating Propensity Scores ---")

X_ps = sm.add_constant(df_analysis[controls].astype(float))
y_ps = df_analysis['high_sentiment'].astype(float)

ps_model = sm.Logit(y_ps, X_ps).fit(disp=0)
df_analysis['propensity_score'] = ps_model.predict(X_ps)

print(f"Propensity Score Model Pseudo R²: {ps_model.prsquared:.4f}")

# Step 3: Nearest Neighbor Matching (without replacement)
print("\n--- Step 2: Matching High and Low Sentiment Firms ---")

treated = df_analysis[df_analysis['high_sentiment'] == 1].copy()
control = df_analysis[df_analysis['high_sentiment'] == 0].copy()

# Simple nearest neighbor matching
matched_pairs = []
used_control_indices = set()

for idx, t_row in treated.iterrows():
    t_ps = t_row['propensity_score']

    # Find nearest control not yet used
    best_match = None
    best_distance = float('inf')

    for c_idx, c_row in control.iterrows():
        if c_idx in used_control_indices:
            continue
        distance = abs(t_ps - c_row['propensity_score'])
        if distance < best_distance:
            best_distance = distance
            best_match = c_idx

    if best_match is not None and best_distance < 0.1:  # Caliper = 0.1
        matched_pairs.append({
            'treated_idx': idx,
            'control_idx': best_match,
            'treated_ps': t_ps,
            'control_ps': control.loc[best_match, 'propensity_score'],
            'treated_strategy': t_row[dv],
            'control_strategy': control.loc[best_match, dv],
            'ps_distance': best_distance
        })
        used_control_indices.add(best_match)

matched_df = pd.DataFrame(matched_pairs)
print(f"Successfully matched pairs: {len(matched_df)}")
print(f"Average PS distance: {matched_df['ps_distance'].mean():.4f}")

# Step 4: Calculate Average Treatment Effect on Treated (ATT)
print("\n--- Step 3: Calculating Treatment Effect ---")

matched_df['strategy_diff'] = matched_df['treated_strategy'] - matched_df['control_strategy']

att = matched_df['strategy_diff'].mean()
att_se = matched_df['strategy_diff'].std() / np.sqrt(len(matched_df))
att_t = att / att_se
att_p = 2 * (1 - stats.t.cdf(abs(att_t), df=len(matched_df)-1))

print(f"\nAverage Treatment Effect on Treated (ATT):")
print(f"  ATT = {att:.4f} {sig_stars(att_p)}")
print(f"  SE  = {att_se:.4f}")
print(f"  t   = {att_t:.3f}")
print(f"  p   = {att_p:.4f}")

# Step 5: Balance check
print("\n--- Step 4: Covariate Balance Check ---")

print("\nBefore Matching:")
print(f"{'Variable':<25} {'Treated':>12} {'Control':>12} {'Std Diff':>12}")
print("-" * 65)

for var in controls:
    t_mean = treated[var].mean()
    c_mean = control[var].mean()
    pooled_std = np.sqrt((treated[var].var() + control[var].var()) / 2)
    std_diff = (t_mean - c_mean) / pooled_std if pooled_std > 0 else 0
    print(f"{var:<25} {t_mean:>12.4f} {c_mean:>12.4f} {std_diff:>12.4f}")

# After matching balance
print("\nAfter Matching:")
matched_treated = df_analysis.loc[matched_df['treated_idx']]
matched_control = df_analysis.loc[matched_df['control_idx']]

print(f"{'Variable':<25} {'Treated':>12} {'Control':>12} {'Std Diff':>12}")
print("-" * 65)

balance_improved = 0
for var in controls:
    t_mean = matched_treated[var].mean()
    c_mean = matched_control[var].mean()
    pooled_std = np.sqrt((matched_treated[var].var() + matched_control[var].var()) / 2)
    std_diff = (t_mean - c_mean) / pooled_std if pooled_std > 0 else 0
    print(f"{var:<25} {t_mean:>12.4f} {c_mean:>12.4f} {std_diff:>12.4f}")
    if abs(std_diff) < 0.1:
        balance_improved += 1

print(f"\nVariables with |Std Diff| < 0.1 after matching: {balance_improved}/{len(controls)}")

ENDOGENEITY TESTS: PSM AND GMM
Sample size: 488
Number of firms: 45

METHOD 1: PROPENSITY SCORE MATCHING (PSM)

Logic:
1. Divide firms into High vs Low sentiment groups
2. Match similar firms based on observable characteristics
3. Compare strategy outcomes between matched pairs
4. This controls for selection bias based on observables

Median sentiment: 0.1818
High sentiment group: 241 obs
Low sentiment group: 247 obs

--- Step 1: Estimating Propensity Scores ---
Propensity Score Model Pseudo R²: 0.0156

--- Step 2: Matching High and Low Sentiment Firms ---
Successfully matched pairs: 223
Average PS distance: 0.0063

--- Step 3: Calculating Treatment Effect ---

Average Treatment Effect on Treated (ATT):
  ATT = 0.1614 **
  SE  = 0.0771
  t   = 2.094
  p   = 0.0374

--- Step 4: Covariate Balance Check ---

Before Matching:
Variable                       Treated      Control     Std Diff
-----------------------------------------------------------------
size_lag1                      10.6

In [38]:
# ============================================
# METHOD 2: SYSTEM GMM (Arellano-Bond / Blundell-Bond)
# ============================================
print("\n" + "=" * 80)
print("METHOD 2: SYSTEM GMM (Dynamic Panel)")
print("=" * 80)


# Prepare data for GMM
df_gmm = df_analysis.sort_values(['ticker', 'year']).copy()

# Create lagged strategy
df_gmm['strategy_lag1'] = df_gmm.groupby('ticker')[dv].shift(1)
df_gmm['strategy_lag2'] = df_gmm.groupby('ticker')[dv].shift(2)

# Create differences
df_gmm['strategy_diff'] = df_gmm.groupby('ticker')[dv].diff()
df_gmm['strategy_lag1_diff'] = df_gmm.groupby('ticker')['strategy_lag1'].diff()
df_gmm['sentiment_diff'] = df_gmm.groupby('ticker')[iv_lag].diff()

for c in controls:
    df_gmm[f'{c}_diff'] = df_gmm.groupby('ticker')[c].diff()

# Drop NAs
df_gmm_clean = df_gmm.dropna(subset=['strategy_diff', 'strategy_lag2', 'sentiment_diff'])

print(f"Observations for GMM: {len(df_gmm_clean)}")

# ---- Difference GMM (Arellano-Bond style) ----
print("\n--- Difference GMM (Arellano-Bond) ---")

# First stage: Use lagged levels as instruments for differenced equation
# ΔY_t = β1 * ΔY_{t-1} + β2 * ΔX_t + ε
# Instruments: Y_{t-2}, X_{t-2}

# Simplified 2SLS approximation of GMM
# Instrument: strategy_lag2 for strategy_lag1_diff

# First stage: regress strategy_lag1_diff on instruments
print("\nFirst Stage: Instrumenting ΔStrategy_{t-1} with Strategy_{t-2}")

X_first = sm.add_constant(df_gmm_clean[['strategy_lag2']].astype(float))
y_first = df_gmm_clean['strategy_lag1_diff'].astype(float)

first_stage = sm.OLS(y_first, X_first).fit()
print(f"  First stage F-statistic: {first_stage.fvalue:.2f}")
print(f"  First stage R²: {first_stage.rsquared:.4f}")

# Get predicted values
df_gmm_clean['strategy_lag1_diff_hat'] = first_stage.predict(X_first)

# Second stage
print("\nSecond Stage: ΔStrategy_t on predicted ΔStrategy_{t-1} and ΔSentiment")

control_diffs = [f'{c}_diff' for c in controls]
X_second = df_gmm_clean[['strategy_lag1_diff_hat', 'sentiment_diff'] + control_diffs].astype(float)
X_second = sm.add_constant(X_second)
y_second = df_gmm_clean['strategy_diff'].astype(float)

second_stage = sm.OLS(y_second, X_second).fit(cov_type='cluster',
                                                cov_kwds={'groups': df_gmm_clean['ticker']})

gmm_diff_coef = second_stage.params['sentiment_diff']
gmm_diff_t = second_stage.tvalues['sentiment_diff']
gmm_diff_p = second_stage.pvalues['sentiment_diff']

print(f"\n  ΔSentiment coefficient: {gmm_diff_coef:.4f} {sig_stars(gmm_diff_p)}")
print(f"  t-statistic: {gmm_diff_t:.3f}")
print(f"  p-value: {gmm_diff_p:.4f}")

# ---- System GMM (Blundell-Bond style) ----
print("\n--- System GMM (Blundell-Bond) ---")

print("""
System GMM combines:
1. Difference equation: ΔY_t instrumented by lagged levels
2. Level equation: Y_t instrumented by lagged differences

This is more efficient when the dependent variable is persistent.
""")

# For System GMM, we stack the difference and level equations
# Simplified implementation using available instruments

# Level equation with lagged differences as instruments
df_gmm_clean['sentiment_lag1_diff'] = df_gmm_clean.groupby('ticker')[iv_lag].diff().shift(1)
df_gmm_level = df_gmm_clean.dropna(subset=['sentiment_lag1_diff', 'strategy_lag1'])

print(f"Observations for level equation: {len(df_gmm_level)}")

# Instrument current sentiment with lagged difference
X_first_level = sm.add_constant(df_gmm_level[['sentiment_lag1_diff']].astype(float))
y_first_level = df_gmm_level[iv_lag].astype(float)

first_stage_level = sm.OLS(y_first_level, X_first_level).fit()
print(f"\nFirst stage (level eq) F-statistic: {first_stage_level.fvalue:.2f}")

df_gmm_level['sentiment_hat'] = first_stage_level.predict(X_first_level)

# Second stage level equation
X_second_level = df_gmm_level[['strategy_lag1', 'sentiment_hat'] + controls].astype(float)
X_second_level = sm.add_constant(X_second_level)
y_second_level = df_gmm_level[dv].astype(float)

second_stage_level = sm.OLS(y_second_level, X_second_level).fit(cov_type='cluster',
                                                                  cov_kwds={'groups': df_gmm_level['ticker']})

gmm_sys_coef = second_stage_level.params['sentiment_hat']
gmm_sys_t = second_stage_level.tvalues['sentiment_hat']
gmm_sys_p = second_stage_level.pvalues['sentiment_hat']

print(f"\n  Sentiment coefficient: {gmm_sys_coef:.4f} {sig_stars(gmm_sys_p)}")
print(f"  t-statistic: {gmm_sys_t:.3f}")
print(f"  p-value: {gmm_sys_p:.4f}")

# Lagged DV coefficient
lag_dv_coef = second_stage_level.params['strategy_lag1']
lag_dv_t = second_stage_level.tvalues['strategy_lag1']
lag_dv_p = second_stage_level.pvalues['strategy_lag1']

print(f"\n  Strategy_{'{t-1}'} coefficient: {lag_dv_coef:.4f} {sig_stars(lag_dv_p)}")
print(f"  t-statistic: {lag_dv_t:.3f}")
print(f"  p-value: {lag_dv_p:.4f}")



METHOD 2: SYSTEM GMM (Dynamic Panel)

Logic:
1. Your data has a lagged dependent variable (strategy persistence)
2. Standard OLS/FE with lagged DV is biased (Nickell bias)
3. GMM uses lagged levels/differences as instruments
4. This addresses endogeneity in dynamic panel settings

Reference: Arellano & Bond (1991), Blundell & Bond (1998)

Observations for GMM: 398

--- Difference GMM (Arellano-Bond) ---

First Stage: Instrumenting ΔStrategy_{t-1} with Strategy_{t-2}
  First stage F-statistic: 160.49
  First stage R²: 0.2884

Second Stage: ΔStrategy_t on predicted ΔStrategy_{t-1} and ΔSentiment

  ΔSentiment coefficient: 0.0130 
  t-statistic: 0.116
  p-value: 0.9073

--- System GMM (Blundell-Bond) ---

System GMM combines:
1. Difference equation: ΔY_t instrumented by lagged levels
2. Level equation: Y_t instrumented by lagged differences

This is more efficient when the dependent variable is persistent.

Observations for level equation: 352

First stage (level eq) F-statistic: 8.39

 

In [39]:
# ============================================
# SUMMARY TABLE
# ============================================
print("\n" + "=" * 80)
print("SUMMARY: ENDOGENEITY TESTS")
print("=" * 80)

print(f"""
══════════════════════════════════════════════════════════════════════════════════════════
Table: Addressing Endogeneity - PSM and GMM Results
══════════════════════════════════════════════════════════════════════════════════════════
                                                    Coef.        t-stat     p-value
──────────────────────────────────────────────────────────────────────────────────────────
Panel A: Propensity Score Matching

  Treatment: High Sentiment (above median)
  Matched pairs: {len(matched_df)}
  Average Treatment Effect (ATT)                   {att:>7.4f}{sig_stars(att_p):<3}      {att_t:>6.2f}      {att_p:.4f}

Panel B: GMM Estimation

  Difference GMM (Arellano-Bond style)
    ΔSentiment                                     {gmm_diff_coef:>7.4f}{sig_stars(gmm_diff_p):<3}      {gmm_diff_t:>6.2f}      {gmm_diff_p:.4f}

  System GMM (Blundell-Bond style)
    Sentiment (instrumented)                       {gmm_sys_coef:>7.4f}{sig_stars(gmm_sys_p):<3}      {gmm_sys_t:>6.2f}      {gmm_sys_p:.4f}
    Strategy_{'{t-1}'}                                  {lag_dv_coef:>7.4f}{sig_stars(lag_dv_p):<3}      {lag_dv_t:>6.2f}      {lag_dv_p:.4f}

──────────────────────────────────────────────────────────────────────────────────────────
Notes: PSM uses nearest-neighbor matching with caliper = 0.1. GMM uses lagged levels/
differences as instruments following Arellano & Bond (1991) and Blundell & Bond (1998).
Standard errors clustered by firm. *** p<0.01, ** p<0.05, * p<0.10.
══════════════════════════════════════════════════════════════════════════════════════════
""")



SUMMARY: ENDOGENEITY TESTS

══════════════════════════════════════════════════════════════════════════════════════════
Table: Addressing Endogeneity - PSM and GMM Results
══════════════════════════════════════════════════════════════════════════════════════════
                                                    Coef.        t-stat     p-value
──────────────────────────────────────────────────────────────────────────────────────────
Panel A: Propensity Score Matching

  Treatment: High Sentiment (above median)
  Matched pairs: 223
  Average Treatment Effect (ATT)                    0.1614**         2.09      0.0374

Panel B: GMM Estimation

  Difference GMM (Arellano-Bond style)
    ΔSentiment                                      0.0130           0.12      0.9073

  System GMM (Blundell-Bond style)
    Sentiment (instrumented)                        0.4219           0.64      0.5214
    Strategy_{t-1}                                   0.3575***        4.60      0.0000

───────────────